![AIAP Banner](images/AIAP-Banner.png)

<h1><center>Designing a Good ML System</center></h1>
<center>Recommended time: approximately <b>2 days</b> on this notebook - the rest of the week is for presenting and defending your design, and (as an optional stretch) building it.</center>
<br>

<font color=darkblue><h3>**Name of Apprentice: team to complete**</h3></font>
<font color=darkblue><h3>**AIAP Email: team to complete**</h3></font>

<a id="helion-context"></a>
# Group 8: Helion diagnostic selection and scheduling

**Status:** design submission revised 21 September 2026 for the user-requested broader objective. The original data audit remains dated 18 September 2026. No Helion model has been trained, no pipeline or service deployed, and no diagnostic savings measured. The original course explanation and questions are retained; authored answer cells are Markdown. The inherited leakage-demo code remains unexecuted.

**Read in this order:** this context, the answers in §§1–9, the [§10.2 blueprint](#helion-blueprint), then the [evidence](#helion-evidence), [procedure policy](#helion-policy), [rolling-horizon scheduling](#helion-scheduling), [data](#helion-data), [evaluation](#helion-evaluation), [operations](#helion-operations) and [worked cases](#helion-worked-cases). For the completed decision walkthrough and steps 1–4, read [Appendix I](#helion-workflow-review). The [pre-read](PRE_READ_TEMPLATE.md), [presentation](presentation_outline.md) and [viva guide](viva_preparation.md) summarise these decisions.

**Authority:** [client brief](../problem-statement/helion_semiconductor_client_brief.md) for the scenario; [rubric](../problem-statement/PRESENTATION_RUBRIC_APPRENTICE.md) for assessment; [original course notebook](../../../all-assignments/assignment8-ml_systems/P1_ml_systems.ipynb), §10.2/cell `2dcc0b8d`, for blueprint and pre-read requirements. Teaching content and images are adapted from that supplied notebook. Source factual statements about the simulated cohort to its files rather than treating the brief as a data export.

## Context to maintain as a new engineer

**Broader objective:** minimise the expected total operational cost of completing diagnostically adequate investigations across rejected stacks, subject to diagnostic completeness, safety and evidence preservation, resource feasibility, and approved turnaround and maximum-wait constraints. Report turnaround separately rather than claiming simultaneous unconstrained minimisation of cost and time.

The integrated design has **one predictive model and two coordinated operational decisions**: eligible next diagnostic alternatives for each stack, then assignment of those procedures to resources and time slots across stacks. The quality engineer approves procedure choice and closure; a lab coordinator or designated shift lead approves dispatch. These responsibilities are proposed, not evidence of an existing staffing arrangement. A deterministic scheduler adds no second learned model.

**Scope provenance:** this broader design follows the user's revision request on 21 September 2026. The original [client brief](../problem-statement/helion_semiconductor_client_brief.md#constraints--scope) restricts the assignment to one model and one decision. We preserve that source and explicitly identify the expansion; course/client acceptance of the wider scope has not been recorded. Mandatory acceptance testing and the brief's scrap disposition still apply. [Business context](../problem-statement/helion_semiconductor_client_brief.md#business-context).

| Person or team | What we need to understand | Proposed responsibility |
|---|---|---|
| Quality manager | What counts as complete enough; consequences of omissions | Approve completeness limits, evaluation protocol and pilot decision |
| On-duty quality engineer | Actual procedure order, exceptions, night-shift needs | Select tests, record findings and overrides, authorise closure |
| Lab coordinator or designated shift lead | Equipment/staff calendars, queues, commitments and urgency | Confirm dispatch, preserve reservations and resolve blocked work; proposed responsibility, not a new hire |
| Process engineering | How confirmed mechanisms inform process investigations | Interpret findings without treating a model association as a recipe cause |
| Yield Engineering | Existing statistics/JMP/Python capability and support capacity | Own data, candidate model, monitoring and investigation of alerts |
| Fab IT and qualification reviewers | Approved software environment, release and recovery procedure | Qualify installation, job health, access and rollback |

The brief names the quality engineer, process-engineering consequence, four-person Yield Engineering owner and fab constraints; the allocation above is a proposed operating agreement. [Helion brief](../problem-statement/helion_semiconductor_client_brief.md)

**Acceptance testing** determines required pass/fail status. **Additional diagnosis** confirms or excludes fault mechanisms after rejection. **Process root-cause investigation** asks why those mechanisms occurred. These are separate decisions in this design. [Helion brief](../problem-statement/helion_semiconductor_client_brief.md)

## Repeatable context-update method

1. Name the next decision, owner, outcome and consequence of being wrong.
2. Map the people and follow ordinary and exceptional cases through the current workflow.
3. Log claims with source, date, applicability and status: brief-stated, verified in files, proposed assumption or unknown.
4. Prioritise unknowns whose answers could change the recommendation.
5. Gather the missing evidence and explain the interpretation back to the relevant owner.
6. Decide at the strength of evidence available, record the rationale and review trigger, and update dependent decisions when facts change.

For Helion, the first learning requests are the actual SOP and closure standard, procedure-level evidence definitions, effort records, audit selection/completion logs, equipment/staff availability, setup/batching rules, deadlines and the qualified deployment environment. This is a proposed inquiry list; no stakeholder interviews or approvals have occurred.

**Operating-assumption update:** [Appendix I](#helion-workflow-review) now uses [SYN-OPS-001](../synthetic-data-assumptions/operation-assumptions/synthetic_operating_assumptions.md) and [MOCK-ENG-001](mock_engineering_inspection_rules.md) for costed paper decisions and staff-feasible slots. These are teaching assumptions, not new operational evidence.


**Welcome back.**

It's been a while. Since the Intro to MLOps session, you've spent time getting your hands dirty with the actual craft of machine learning, wrangling data, engineering features, training models, and tuning them until the numbers finally looked good in your notebook. And somewhere in there, a nagging question may have started to form: *this all lives in my notebook, on my machine. How would I ever hand it to a client who has never opened Jupyter in their life, and have it keep working without me sitting next to it?*

That question is exactly what this notebook is about. A model that only runs when you're there to run it isn't a product, it's a demo. Here we zoom out from "train a model" to the whole machine around it: the **end-to-end (E2E) ML system** that takes a business problem, turns it into a model running reliably in production, and keeps it working long after you've stopped babysitting it.

The recurring question across every section is deliberately simple:

> **What makes a pipeline _good_, and how do we construct one?**

We won't answer that with a single checklist. A "good" system is one whose design *fits its constraints*, and those constraints change with every problem. So instead of memorising the one right answer, you'll build the judgement to reason toward a good answer for whatever problem you're handed.

### **What we'll cover**

- **What makes a pipeline "good"** - the cross-cutting qualities we judge *every* stage against
- **The ML system lifecycle** - the E2E map, and why it's a loop, not a line
- **Each stage in turn** - problem framing → data → training → evaluation → deployment → monitoring, and what "good" means for each
- **Putting it together** - reasoning about trade-offs and designing a system that fits its constraints
- **From understanding to building** - turning that design into a one-page blueprint and directing an LLM to build the pipeline

By the end, you should be able to:

- Explain what distinguishes a well-designed ML pipeline from one that merely runs
- Map any ML problem onto the full lifecycle and identify the key design decisions at each stage
- Reason about the trade-offs between competing "good" choices under real-world constraints
- Turn your design into a blueprint and direct - and *verify* - an LLM building the pipeline from it

### **How to use this notebook**

This notebook assumes the hands-on MLOps foundations from the earlier **Intro to MLOps** week (data handling, config, experiment tracking, containers). Here we focus on the *design judgement* that ties those pieces into a good end-to-end system.

Throughout, you'll be asked to reason about a **concrete case study scenario** - either the one you've been assigned for this deep-skilling topic, or the problem you brought and had approved (see `BYO_PROBLEM.md`). Keep it open beside you and **answer every question against it.**

The questions are written to apply to *any* ML problem on purpose: the whole skill being trained here is mapping general principles onto *your specific* problem. A generic, scenario-free answer earns little - the reasoning about a real, concrete problem is the point.

That reasoning is also your **main deliverable**. At the end of the week you'll give a **short presentation** of your design - that part is *expected* of everyone - but the **key thing you're assessed on is the oral defence**: the live Q&A where mentors push on your choices ("why not the alternative?", "what breaks if…?") and you justify them on the spot. Present it *clearly* - a clean walkthrough helps your defence land - but you're graded on the reasoning, not the deck's looks.

**This notebook is self-contained.** Working through it, and producing the design it leads you to (§10), is the **core** of the week - *building* that pipeline is an **optional stretch** (§10.3): never required, and worth only a **small bonus (~1-2% of the grade)** if you get it running as intended. Two *optional* companions sit alongside it:

- a **deep-dive** notebook - six advanced "lenses" for teams who want to make a solid design genuinely good; **not required**, and
- a **deployment reference** - the thinking for taking your pipeline all the way to a running service, an optional **bonus**.

Neither is needed to complete the core: this notebook plus the design you produce from it stand on their own.

> Wherever a question says **"your case study,"** it means that scenario - assigned or self-chosen.

# **1 What Makes a Pipeline "Good"?**

Before we walk through the lifecycle, we need a vocabulary for *quality*. A pipeline that "works" on your laptop today is not the same as a **good** pipeline. Plenty of models that scored well in a notebook never made it to production, or made it there and quietly broke.

The difference is a set of **cross-cutting qualities**. They are cross-cutting because they don't belong to any single stage, they apply to the data pipeline, the training pipeline, and the serving pipeline alike. Keep them in mind as a lens for every section that follows.

| Quality | The question it answers | Why it matters |
|---|---|---|
| **Reproducibility** | If I run this again, or you run it, do we get the same result? | Without it, you can't debug, can't audit, and can't trust that the model in production is the one you evaluated. |
| **Modularity** | Can I change one part without breaking the others? | Swapping a model, a data source, or a feature shouldn't mean rewriting the whole thing. Separation of concerns keeps a system maintainable. |
| **Configurability** | Can I change *what* it does without editing the code that does it? | Experiments (different models, features, parameters) should be driven by config, not by commenting out lines. |
| **Automation & orchestration** | Can it run E2E without someone babysitting each step? | Manual steps don't scale and don't survive the person who knows them leaving. |
| **Testability & validation** | How do I know it's still correct after a change? | Data changes silently, code changes deliberately. Both need checks that fail loudly. |
| **Observability** | When it breaks in production, will I know, and will I know *why*? | You can't fix what you can't see. Logging, metrics, and monitoring make failures visible. |
| **Scalability** | Does it still work when the data is 100× bigger, or the traffic 100× heavier? | The volume that fits today rarely fits tomorrow. |
| **Reliability & reproducible environments** | Does it behave the same on my machine, in CI, and in production? | "Works on my machine" is where a huge share of production failures are born. |

Notice that these qualities often **pull against each other**. A system that is maximally configurable can become hard to understand; one tuned for massive scale is often more complex and harder to reproduce cheaply. There is no configuration that maxes out all of them at once.

That tension is the heart of good design. Good design is not about scoring 10/10 on every quality, it's about knowing *which ones matter most for this problem* and spending your complexity budget there.

Pick two qualities from the table above and describe a concrete situation where improving one would **hurt** the other.

**Helion answer: reliability versus flexibility.**

We could let every shift engineer change test-ranking weights. That would make the tool adaptable, but two engineers could then receive different recommendations for the same case without a qualified change. I propose versioned settings, reviewed releases and an override-with-reason instead. The trade-off is slower tuning in exchange for a recommendation we can reproduce during an investigation review. This fits the brief's rotating shifts, small ownership team and change-control requirement. [Helion brief](../problem-statement/helion_semiconductor_client_brief.md)

Our priorities are testability/validation, reliability and observability. We deliberately spend less effort on scaling far beyond this cohort. Those are design choices, not measured comparisons.

**(Optional)** Think back to something you've built - a personal project, or the Technical Assessment you submitted to join AIAP. Which of these qualities did it have, and which did it lack? What would have broken if someone else had to pick it up and run it six months later?

**Optional personal reflection: for the team to complete.**

No personal project history was supplied. Each member should name a real example, identify one missing engineering quality and explain a concrete handover failure. We do not substitute an invented experience for this reflection.

# **2 The ML System Lifecycle**

People often picture machine learning as "train a model." In a real system, training is a small box in the middle of a much longer loop. The bulk of the work, and the bulk of the failures, live in the stages around it.

Here is the E2E lifecycle we'll use as our map for the rest of the notebook:

<p align="center">
  <a href="images/ml_lifecycle.png" target="_blank" rel="noopener">
    <img src="images/ml_lifecycle.png" alt="An ML project Lifecycle" width="100%" style="max-width:900px;">
  </a>
  <br>
  <sub><em>Click the diagram to open it full-size in a new tab.</em></sub>
</p>

Two things are worth internalising immediately:

- **It's a loop, not a line.** A model is never "done." Production teaches you things your training data never showed you, and those lessons flow back to reshape the problem, the data, and the next iteration. A system designed as a straight line has no answer for "what happens after launch?"
- **Every arrow is a handoff, and handoffs are where systems break.** The interesting failures are rarely inside a box; they're at the boundaries, the schema that changed upstream, the feature computed one way in training and another way in serving, the metric that looked great offline and tanked live.

Why do you think most ML *project* failures happen outside the "train a model" box? Where would you bet the failures cluster, and why?

**Helion answer: inspect the handoffs.**

For this case, I would first investigate the boundary between a recommendation and the next procedure: a score is useful only if it changes an investigation appropriately. I would also inspect label creation. Procedures chosen by the engineer determine which mechanisms receive findings, so the resulting training data can reproduce the old selection policy. These are scenario-specific failure hypotheses, not an estimate of how often ML projects fail generally. [Helion brief](../problem-statement/helion_semiconductor_client_brief.md)

The supplied files make the issue concrete: they have simulated fault labels but lack diagnostic history and effort. A technically correct classifier alone cannot establish operational savings. [repository task boundary](../README.md#proposed-cohort-labels-and-inputs)

Explain the loop in your own words: give one concrete example of something you'd only discover *after* deployment that would force you to revisit an *earlier* stage.

**Helion answer: the loop changes what we must collect.**

Illustrative future observation: engineers follow the ranking and order fewer acoustic examinations. Recorded delamination positives fall. That could reflect fewer delamination faults, fewer opportunities to observe them, or both. We compare independently selected full-battery cases with routine cases before deciding. If the difference is ascertainment, revisit label collection and evaluation rather than blindly retraining on the routine labels. The brief supplies the selective-testing risk and independent 1-in-20 audit. [Helion brief](../problem-statement/helion_semiconductor_client_brief.md)

Record the observation, affected assumption, evidence needed and decision owner. The loop connects operational findings back to framing, data and evaluation.

# **3 Problem Framing & Requirements**

Everything downstream inherits the quality of this stage. The most technically flawless pipeline is worthless if it solves the wrong problem, and no amount of model tuning rescues a badly framed one.

Almost every real ML project begins the same way: a stakeholder describes a business pain and essentially says *"We'd like to use AI to fix it. Can your team build us something?"* That kind of request - and it is almost certainly how your case study scenario is framed - is **not** a problem statement. It's a vague business wish. Turning it into something a machine can learn is your job, and it is a design skill, not a modelling one.

### **From a business ask to an ML problem**

A good framing pass answers, at minimum:

| Dimension | The question to force an answer to |
|---|---|
| **Objective** | What decision or action will this model actually inform? Who acts on its output? |
| **ML task type** | Is this classification, regression, ranking, forecasting, clustering, recommendation...? Or does it even need ML at all? |
| **Target & label** | What exactly are you predicting, and where does the ground-truth label come from? Is it available at training time *and* reliably at prediction time? |
| **Success metric** | How will you know it worked, in *business* terms, not just model terms? What's the baseline to beat? |
| **Constraints** | Latency, budget, interpretability, fairness, regulatory limits, available data, team skills. |
| **Cost of being wrong** | Is a false positive worse than a false negative? By how much? This shapes everything downstream. |

### **What the model actually outputs**

The framing table asks *what* you predict; be just as deliberate about the *shape* of the output, because that too is a design decision, not a given.

- **It's often not a single yes/no.** Many real systems output one of *several decision tiers* - *approve / reject / send to a human*, or *fast-track / auto-reject / escalate*. Two classes is just the simplest case; don't force a multi-way decision into a binary one because binary felt like the default.
- **"Hand it to a human" is a first-class option.** When the model exists to *support* an expert rather than replace them, routing the uncertain or high-stakes cases to a person is frequently the highest-value design. A system that confidently automates the easy majority and *escalates* the rest can beat one that insists on deciding everything - and it gives you a natural place to put the cases the model isn't sure about.
- **What you surface can be risk *or* opportunity.** Sometimes you're flagging danger (a likely-fraudulent seller, a possible counterfeit); sometimes you're surfacing upside (the product an audience is about to buy). The systems thinking is identical - you are prioritising scarce human attention - even though one points at trouble and the other at revenue. Everywhere this notebook says "flag," read it as whichever fits your problem.

This shape ripples downstream: it changes your metric (§6), how a human consumes the output (deployment, §7), and what "a mistake" even costs - which is the next thing to pin down.

### **"Do we even need ML?"**

A genuinely good engineer asks this early. ML adds cost, latency, unpredictability, and a maintenance burden. If a handful of business rules, a heuristic, or a simple query solves the problem to an acceptable standard, that is often the *better* system. Reach for ML when the pattern is too complex to specify by hand and you have the data to learn it from.

<details>
<summary><em>A useful sanity check</em></summary>

Before committing to ML, ask: *"If I had a rule-based version of this, how good would it be, and how much better does the ML version need to be to justify its cost?"* That rule-based version is also your **baseline**, the thing your model has to beat to earn its place.

</details>

For your case study scenario, write a **one-sentence problem statement** that names the objective, the ML task type, and the thing being predicted. Then list the two constraints you think would most shape the solution.

**Problem statement.** Select eligible diagnostic procedures for rejected HBM stacks and schedule them across shared resources to minimise expected total completion cost, subject to diagnostic completeness, safety/evidence preservation, resource capacity and approved turnaround constraints. One proposed multi-label logistic model supplies seven initial fault probabilities; a transparent diagnostic policy and deterministic constraint-based scheduler support two coordinated human decisions.

Coexisting faults and selectively observed labels remain central diagnostic constraints. Shared equipment, staffing, prerequisites and uncertain future findings add operational constraints. The last group is part of the user-requested expansion, not a resource model supplied by the original [brief](../problem-statement/helion_semiconductor_client_brief.md#constraints--scope).

The engineer validates procedure choice and closure. The coordinator or shift lead confirms the resource/time allocation. Mandatory acceptance tests continue; the system neither ships material nor repairs the stack. A fault prediction is not a causal finding about a recipe. [Client context](../problem-statement/helion_semiconductor_client_brief.md#business-context).

For your framing, what is the **cost of being wrong** in each direction (a false positive vs a false negative), and how should that asymmetry influence the metric you optimise?

**Helion answer: connect scores to consequences.**

A false-positive mechanism score can prompt an unnecessary procedure. A false-negative score becomes harmful if the workflow omits or delays the test that would expose a real mechanism. Missing a second mechanism can misdirect process engineering. The engineer's procedure and closure decisions therefore determine the realised error cost. [Helion brief](../problem-statement/helion_semiconductor_client_brief.md)

Use operational completion cost as the primary business endpoint, with diagnostic completeness and turnaround requirements as constraints. Until rates and non-overlapping cost definitions are agreed, report engineer, equipment and elapsed time separately and make no scalar total-cost improvement claim. The brief gives procedure durations but no defensible money value for an omitted mechanism, no engineer-labour conversion and no universal FN:FP ratio. Record those gaps. Do not invent a 10:1 penalty. The initial design uses continuous scores to rank eligible procedures and gives scores no authority to exclude a mechanism or close a case. See [evaluation](#helion-evaluation).

**(Optional)** Describe one version of your problem where a **non-ML solution** might be the better system. What would make you change your mind and reach for ML?

**Optional Helion answer: retaining the rules is a valid outcome.**

If inspected warpage/void/delamination and the existing electrical rules already select the necessary procedures efficiently, we keep that workflow. ML must earn its maintenance and qualification effort by improving the same cases at the same completeness requirement. The incumbent already sees the strongest inspection inputs. [Helion brief](../problem-statement/helion_semiconductor_client_brief.md)

Evidence that would change the decision: fresh audited cases and an approved controlled pilot show fewer unnecessary optional procedures or less hands-on effort, with uncertainty small enough to establish the agreed completeness guardrails and positive net value. Better fault-ranking metrics by themselves would not change it. A scheduler may improve dispatch even when the classifier adds no value, so compare these components separately. We may retain incumbent diagnostic rules while adopting a separately qualified scheduling improvement.

# **4 The Data Pipeline (systems view)**

You've met this stage in depth in the DataOps notebook: data sources, shapes, storage, quality tiers (Bronze → Silver → Gold), and ETL vs ELT. Here we look at it through the *quality lens* from Section 1, because the data pipeline is where the "good system" qualities are won or lost first.

Three ideas that separate a good data pipeline from one that merely loads a file:

- **Data validation, not just data loading.** A good pipeline treats incoming data as guilty until proven innocent. Schemas are checked, ranges validated, nulls counted, and violations fail *loudly* rather than silently poisoning the model. (Investigate tools like *Great Expectations* or *Pandera* to see what this looks like in practice.)
- **Reproducibility through versioning.** If you can't say *which* data trained a given model, you can't reproduce or audit it. Data versioning (e.g. *DVC*, or immutable snapshots in the Bronze layer) makes "the exact data behind model v3" a question you can answer.
- **Train/serve consistency.** A feature computed one way in your training notebook and a subtly different way in the live service is one of the most common and most invisible causes of a model that "worked in eval but fails in production." This gap is called **training-serving skew**. The first-line fix is to **use the exact same feature/transform code on both paths** (and, where you can, log the features actually served in production and reuse them for training). When you have many engineered, shared, or low-latency online features, a **feature store** formalises this - computing a feature *once* and serving identical logic to training and inference - but it's *one option, not a requirement*: plenty of systems, and most NLP/CV models (where preprocessing ships bundled with the model), avoid skew with shared code alone.

### **Data concerns that shape the pipeline**

Beyond validation, a few upstream choices quietly shape everything downstream:

- **ETL vs ELT (a callback to DataOps).** *Where* you transform data - before loading (**ETL**) or after, inside the warehouse (**ELT**) - changes where validation lives, what it costs, and how fresh the data can be. ETL transforms up front (clean, governed loads; schema-on-write; but rework when needs change); ELT loads raw and defers transforms (flexible, schema-on-read; but validation and cost move downstream). Neither is "right" - the fit depends on volume, freshness needs, and how often the transforms change.
- **Schema changes & data contracts.** Upstream owners change fields without telling you. A **data contract** - an explicit agreement on schema, types, ranges, and freshness between the producer and your pipeline - turns a silent breakage into a loud, catchable one.
- **Freshness & ingestion mode.** Batch vs streaming ingestion, and how stale the data is allowed to be, is a design decision that ripples straight into your serving pattern (Section 7) and what you monitor (Section 8).

*(The companion deep-dive notebook takes ETL/ELT trade-offs, data contracts, and schema evolution further.)*

Give a concrete example of **training-serving skew**: one feature that could plausibly be computed differently at training time versus prediction time, and what the consequence would be.

**Helion answer: a missingness mismatch.**

Suppose training computes mean copper void percentage over measured constituent dies, preserving a measured-die count, while serving fills unmeasured dies with zero before averaging. A lightly sampled stack then appears to have less voiding in serving. That is a change in the meaning of the feature, even if both columns share a name. Detailed metrology is sampled, and missing reasons require the status field, not the blank alone. [data dictionary](../data/data_dictionary.csv)

Proposed control: use one versioned aggregation function in both paths. A fixture containing a measured value, a true zero and an unmeasured die must produce the same mean, coverage count and missingness flags in both paths. No extra measurement is requested.

Which of the Section 1 qualities does *data validation* most directly protect, and how would you notice its absence only *after* a model was already in production?

**Helion answer: validation protects testability and reliability.**

Proposed checks reject duplicate stack keys, conflicting lot assignments, invalid units, unavailable acceptance records and impossible label states. Expected metrology gaps remain valid inputs under a qualified missing-data path. A schema or timestamp failure makes the affected recommendation unavailable and exposes the SOP fallback. Scheduling also validates resource calendars, staff attendance, non-overlap, current reservations and specimen compatibility. Stale availability disables automated slot promises, with manual dispatch retaining existing commitments.

Without these checks, a join could silently duplicate a rejected stack and make a single lot dominate the training set or reported hours. The source tables have different grains, including dies, lots and stacks, so the joins require explicit keys and aggregation. [Record structure](../README.md#delivered-v1-data-cohort-and-record-structure). See [data design](#helion-data).

# **5 The Training / Model Pipeline**

This is the box most people think of as "the ML." Note what a *good* version of it optimises for: not the single best score you can hand-tune once, but the ability to **run many experiments reproducibly and pick a winner you can trust and rebuild**.

### **What makes a training pipeline good**

- **Config-driven experimentation.** Which model, which features, which hyperparameters, which data split, should all be changeable from a config file or command-line arguments, *not* by editing the code. This is the configurability quality from Section 1, and it's what lets a whole team run comparable experiments. (This is often an explicit project requirement: a pipeline that is *easily configurable to enable rapid experimentation*.)
- **Reproducible runs.** Same config + same data + same seed → same model. That means pinning random seeds, pinning dependency versions, and recording exactly what went into each run.
- **Experiment tracking.** When you've run 40 experiments, memory and filenames won't save you. Tools like **MLflow**, *Weights & Biases*, or *DVC* log each run's parameters, metrics, and artifacts so you can compare them and reproduce the winner.
- **Model versioning & registry.** The trained model is an artifact that needs a version, lineage, and a promotion path (e.g. staging → production). A **model registry** is where a specific model version becomes "the one we deploy."

<br>

![An MLflow run, tracking a training run's parameters and metadata](images/mlflow-register-model.png)

### **Separation of concerns**

Notice how the qualities interlock here. A good training pipeline is **modular**: data loading, preprocessing, feature engineering, model definition, training, and evaluation are separate components with clean interfaces. That modularity is *what makes* configurability possible, you can only swap the model freely if the model isn't tangled into the data-loading code.

This is also exactly why production ML work is built as **`.py` modules and classes**, not a notebook: notebooks encourage tangled, top-to-bottom, run-once code that's hard to reuse, test, or automate. (Expect your project phase to hold you to the same standard.)

Explain *why* config-driven design and modularity reinforce each other. Could you have one without the other? What would that look like?

**Helion answer: configuration selects behaviour, modules implement it.**

Proposed modules would validate records, build features, fit the seven-output model, evaluate policies and produce scores. Separate proposed modules validate resource state, enumerate eligible procedure alternatives, produce a feasible schedule and preserve confirmed reservations. A versioned configuration would select the feature allowlist, regularisation strength, split manifest and procedure catalogue, together with approved cost rates, fault weights, scheduling horizon, freeze window and service constraints. Modules make these choices replaceable without changing the rest of the workflow.

You can have modules with hard-coded choices, but every experiment then needs code edits. You can have a configuration file wrapped around a monolithic script, but changing one setting can still affect unrelated stages. We propose both, with a small explicit configuration surface. This is a future implementation design; no pipeline has been built in this package.

You've run 30 experiments over two weeks and one scored best. List everything you would need to have recorded to **reliably rebuild that exact model** months later. What breaks reproducibility if any one of them is missing?

**Helion answer: a run must be reconstructible.**

For a future experiment record: source snapshot hashes and schema; cohort and lot/time split IDs; label provenance and observation cut-off; feature definitions and code revision; imputation/encoding parameters; all model and policy settings; seed; runtime/library versions; fitted weights; validation outputs by fault and slice; chosen operating policy; rejected alternatives; and the complete release manifest. Missing the label cut-off can admit future information even when the CSV name is unchanged.

For scheduling reproducibility, additionally retain the resource snapshot and timestamp, arrival/priority/deadline records, setup/batch rules, committed jobs, duration estimates, chosen alternatives, solver settings/status, plan version and human overrides. Reconstruct what was known at dispatch, not the final queue after outcomes arrived.

Keep these artifacts on the fab network in an append-only run directory. Do not infer regeneration from the old seed: the cleaned v1 package explicitly lacks its original generator. [Reproducibility limits](../README.md#cleaned-package-layout).

Good ML engineering practice insists the pipeline live in `.py` scripts rather than a notebook. Beyond "they said so," argue the engineering case - what specifically does a notebook make harder?

**Helion answer: separate the written argument from production execution.**

This notebook is a design submission. A future production workflow would use tested modules because the same feature and state-transition logic must run under scheduled scoring, interactive procedure selection, resource scheduling and validation. Explicit function inputs make it easier to discover a dependency than hidden interactive state does.

The notebook remains useful for explanation and review. It would not be the unattended night-shift runtime. The only inherited runnable cell is the course's leakage demonstration, retained without execution or outputs. We have not trained a Helion model.

# **6 Evaluation**

Evaluation is where good judgement most visibly separates from naive practice. A single accuracy number is almost never the whole story, and optimising the wrong metric can lead you, with full confidence, to ship a worse system.

### **Choosing the right metric**

The metric has to reflect the **cost of being wrong** you defined back in Section 3. For imbalanced problems (fraud, defects, rare-but-costly failures), plain accuracy is actively misleading, a model that predicts "no problem" every time can score 99% and be useless. This is why precision, recall, F1, ROC-AUC, PR-AUC, and their trade-offs exist: they let you weigh the two kinds of error separately.

These ideas don't stop at two classes. When the output is several decision tiers (*approve / reject / review*), "cost of being wrong" becomes a **cost matrix** over the tiers - wrongly auto-approving a fraudster is not the same cost as needlessly sending a good seller to review - and you optimise for the errors that actually hurt, not for aggregate accuracy. If the output is a ranking or a prioritisation (which case does the human look at first?), reach for rank-aware metrics like precision@k rather than a single classification score.

And a metric alone doesn't make a decision. A classifier emits a **score**; something has to turn that score into an action, and that something is an **operating threshold**. The default of 0.5 is an arbitrary inheritance, not a choice - it is only right when the two errors cost the same, which for your scenario they almost certainly don't. The threshold is where your cost-of-error asymmetry becomes an actual, tunable knob: move it down and you catch more of the costly cases at the price of more false alarms (and more human-review load); move it up and you spare the reviewers but let costly cases through. Pick it from the *consequences* - the review capacity you actually have, the ratio of the two costs - and be able to say why it sits there. Decision tiers are the same idea with two cut-offs instead of one (auto-approve below, auto-reject above, human review in between), and the band you send to a human is a capacity decision as much as a modelling one.

### **Beyond the headline number**

| Practice | What it protects against |
|---|---|
| **Baselines** | Shipping a complex model that barely beats (or loses to) a trivial heuristic. Always compare against the simplest reasonable alternative. |
| **Honest data splitting** | **Data leakage** - when information from the test set (or the future) sneaks into training, giving a beautiful offline score that collapses in production. |
| **Error analysis** | A single aggregate metric hiding systematic failures on an important slice (e.g. one region, one product category). |
| **Robustness & fairness checks** | A model that performs unevenly across groups, or falls apart on slightly shifted inputs. |
| **Offline vs online evaluation** | Assuming a good offline score guarantees good live behaviour. It doesn't, which is why teams A/B test and monitor after launch. |

<details>
<summary><em>Data leakage: the trap that looks like success</em></summary>

Leakage is dangerous precisely because it makes your metrics look *better*, so nothing alerts you. Classic sources: scaling or imputing using statistics computed over the *whole* dataset before splitting; using a feature that is only known *after* the outcome you're predicting; or letting the same entity appear in both train and test. A model with a suspiciously perfect score deserves suspicion, not celebration.

</details>

**Questions**

Give an example problem where **accuracy is a misleading metric**, and name the metric(s) you'd use instead and why. Then, for *your* case study: roughly where would you set the **operating threshold** (or your tier cut-offs), and what - in consequences, not in model terms - makes 0.5 the wrong answer?

**Helion answer: an accuracy trap and an operating policy.**

Verified synthetic example: predicting “no die crack” for every rejected stack is correct for 869/916 cases, about 94.9%, while identifying none of the 47 die-crack positives. The count is from [synthetic fault labels](../data/simulation_truth/simulation_truth_stack.csv) restricted to rejected IDs in [acceptance labels](../data/ml/ml_stack_labels.csv). This is arithmetic, not a fitted result.

Use per-mechanism recall, average precision and calibration as supporting measures, and operational cost at fixed completeness and approved turnaround constraints for the integrated workflow. Keep resource-time components separate until priced consistently. No 0.5 cutoff controls investigation closure. Initially all unresolved mechanisms stay in scope, whatever their score. Any future score-based test-omission threshold must be chosen on validation under an approved completeness limit and confirmed on fresh cases. We cannot provide an evidence-based numerical omission threshold from these files. [Decision policy](#helion-policy).

Describe one plausible route to **data leakage** in your case study scenario. How would you catch it *before* it fooled you?

**Helion answer: later annotations reveal the answer.**

Joining all columns from the acceptance-test file would also import `defect_type`, `defect_severity` and `final_disposition`, which the brief places after investigation closure. These must not be features. Initial acceptance measurements may be legitimate at this later checkpoint, even though the archived binary task prohibited the same file. [Helion brief](../problem-statement/helion_semiconductor_client_brief.md) [repository task boundary](../README.md#proposed-cohort-labels-and-inputs)

Proposed checks use a field allowlist, feature-availability timestamps and a negative test that deliberately includes one forbidden column. Also exclude `timing_margin_ps` initially: in this simulation its sign depends on electrical pass/fail. Fit preprocessing using training rows only and preserve lot/time separation. [leakage guidance](https://scikit-learn.org/stable/common_pitfalls.html#data-leakage)

A model scores brilliantly offline but disappoints once live. List three distinct reasons this can happen, mapping each to an earlier lifecycle stage.

**Helion answer: three different failures.**

1. **Data:** imputation uses a different denominator at serving, shifting sampled-die aggregates. Fix shared preprocessing and fixtures.
2. **Evaluation:** selectively untested mechanisms became negatives, so the model's apparently good score measures which tests people chose. Fix provenance and independent audit evaluation.
3. **Framing/operation:** higher fault-ranking metrics do not change optional work or costs. Alternatively, stale queue information causes a feasible-looking plan to miss deadlines. Evaluate both layers separately. Reordering identical procedures cannot reduce fixed execution time, but scheduling may reduce waiting or sequence-dependent setups; measure those outcomes rather than crediting the classifier.

These are proposed failure scenarios grounded in the brief's sampling, selective labels and savings constraints. [Helion brief](../problem-statement/helion_semiconductor_client_brief.md) They are not observed results from an implemented system.

# **7 Deployment & Serving**

A model that only runs in your notebook delivers zero value. Deployment is the stage that puts it where decisions are actually made, and it introduces a whole new set of constraints that never mattered during training.

### **How models are served**

The right serving pattern is driven by *how the predictions get used*:

| Pattern | When it fits | Example |
|---|---|---|
| **Batch / offline** | Predictions can be precomputed on a schedule; no need for instant answers. | Score every customer's churn risk nightly and write it to a table. |
| **Online / real-time (API)** | A prediction is needed on demand, within a tight latency budget. | A tool asking "how risky is *this* case?" the moment it's created. |
| **Streaming** | Predictions react to a continuous flow of events as they arrive. | Flagging anomalous transactions the moment they happen. |
| **Edge / on-device** | Latency, privacy, or connectivity rule out a round-trip to a server. | A model running on a phone or sensor. |

### **Reproducible environments**

Remember "works on my machine"? Deployment is where that bites hardest. The fix is to ship the model *with its environment*, pinned dependencies, and increasingly **containers** (Docker) so the exact same environment runs in development, testing, and production. This is the reliability quality from Section 1 made concrete.

<br>

![Containers vs virtual machines](images/docker-vs-vm.png)

For your problem from Section 3, which serving pattern fits, and *why*? What would have to change about how the prediction is used for a different pattern to become the right choice?

**Helion answer: nightly probabilities, current local resource decisions.**

Keep nightly validated MES snapshots for initial fault scoring. The diagnostic policy updates eligible alternatives from current findings. The scheduler uses a separate qualified local feed for resource status, staff availability, queues and reservations, updating at relevant events within the air-gapped fab. These scheduling inputs are proposed additions; the brief only establishes nightly manufacturing consolidation. [Fab constraints](../problem-statement/helion_semiconductor_client_brief.md#business-context).

New results, arrivals, outages, overruns or urgency changes replan uncommitted work on a bounded rolling horizon. Started work and confirmed near-term reservations remain protected except through authorised exception handling. Nightly initial probabilities are not recalculated as posterior probabilities after every test.

Missing initial data uses diagnostic SOP. Stale resource data or a scheduling failure means manual dispatch without new automated slot promises. Fresh scheduler state cannot repair missing diagnostic data, and faster inference cannot repair an unavailable MES snapshot. See [scheduling](#helion-scheduling).

What new constraints appear at deployment that were irrelevant during training? Name at least three.

**Helion answer: deployment adds operational obligations.**

The fab cannot download packages or call cloud services at runtime. An engineer needs a usable fallback at 3am. Releases touching material decisions require change control and qualification. These are constraints in the client brief. [Helion brief](../problem-statement/helion_semiconductor_client_brief.md)

Proposed responses: package the approved runtime and dependencies offline; show snapshot/model/policy versions and unresolved mechanisms; retain a tested rollback; persist investigation state and resource commitments across shifts; and define diagnostic SOP plus manual-dispatch fallback. Fab IT and the quality owner must qualify the target environment before a pilot. Four engineers can share these responsibilities; the design does not assume four additional hires.

In your own words, why does containerisation address the "works on my machine" problem more thoroughly than simply writing down a `requirements.txt`? (Investigate what a container actually packages.)

**Helion answer: a container captures more of the runtime.**

A Python requirements file describes Python packages. A container image can additionally package the interpreter, application files and user-space system libraries. Containers still depend on a compatible host kernel and runtime, so they do not guarantee identical hardware behaviour or replace qualification. [Docker: what a container contains](https://docs.docker.com/get-started/docker-concepts/the-basics/what-is-a-container/).

Proposed Helion release: a qualified CPU runtime image identified by digest, model and preprocessing artifacts, signed-off configuration and an offline rollback bundle. Image approval by fab IT is a prerequisite. The diagram specifies the release boundary without claiming that a container or service has been built.

# **8 Monitoring & Feedback**

Launch is the beginning, not the end. The world the model was trained on keeps changing, and a model that was excellent at launch silently decays. Monitoring is how you *notice*, and the feedback loop is how you *respond*. This is the arrow that closes the loop in Section 2.

### **What decays, and why**

- **Data drift** - the distribution of inputs shifts away from what the model saw in training (a new market opens, customer behaviour changes, a sensor is recalibrated). The model is still doing what it learned; the world just moved.
- **Concept drift** - the *relationship* between inputs and the target changes (what predicted the outcome last year no longer does). Even a perfect model of the old world is now wrong.
- **Data quality breakages** - an upstream schema change, a broken feed, a field that starts arriving as null. Often sudden, and often the first thing to actually break.

But **monitoring is more than drift detection.** A model can be perfectly calibrated and still deliver nothing because the *pipeline feeding it* died overnight, or the serving API is timing out. A good setup watches the whole system, not just the model.

### **A good monitoring setup watches several layers**

| Layer | What you watch | Example signal |
|---|---|---|
| **Inference / service** | Is the serving path healthy? | Latency (p50/p99), error rate, throughput, saturation |
| **Pipeline** | Did the data & training jobs actually run and succeed? | Job success/failure, run duration, **data freshness**, row-count/volume, upstream availability |
| **Data** | Are the inputs still what we expect? | Feature distributions, null rates, schema conformance |
| **Model** | Are the predictions still good? | Prediction distribution, and, once labels arrive, live accuracy vs offline |

*("Operational" monitoring is really the first two - **service and pipeline** health - not a single "is it up?" check.)*

The hard part of the model layer: **ground-truth labels usually arrive late** (you often learn the true outcome only well after the prediction was made). So monitoring often has to lean on *proxy* signals in the meantime. When drift or decay crosses a threshold, that becomes the trigger to retrain, and ideally that retraining is automated, closing the loop back to Section 5.

### **From monitoring to alerting**

Monitoring only *observes* - a dashboard nobody watches catches nothing. The other half is **alerting**: thresholds that actively page a human or trigger an action. For each signal, decide what fires it (error rate over *X*, a pipeline job failing, freshness past its SLA, a null-rate spike), how severe it is, **who owns the response**, and whether it's handled by a person or automatically (a drift alarm crossing its threshold is exactly the automated retrain trigger above). The craft is tuning thresholds so alerts stay *actionable*: too many and people tune them out (**alert fatigue**), too few and failures run silently for weeks.

Distinguish **data drift** from **concept drift** with a concrete example of each for your case study scenario. Would the *same* monitoring signal catch both?

**Helion answer: changing inputs versus changing relationships.**

Data-drift example: the new bonder changes the distribution of alignment offsets or introduces an unseen tool ID. Concept-drift hypothesis: with the new underfill supplier, the same measured void percentage corresponds to a different confirmed-fault distribution. The impending changes are brief-stated; their precise effects remain hypotheses. [Helion brief](../problem-statement/helion_semiconductor_client_brief.md)

Input monitoring can flag the first and suggest investigation of the second, but only appropriately observed outcomes can establish a change in the input-to-fault relationship. Monitor supplier/tool provenance outside the fitted feature set as needed. A material batch ID alone does not establish supplier identity.

Ground-truth labels arrive late in most real problems. What **proxy signals** could you monitor in the meantime to get an early warning that your model is degrading?

**Helion answer: early warnings without pretending they are labels.**

Proposed proxies are import success, required-record availability, unexpected missingness, product/tool mix, score distribution, inconclusive procedure frequency, override reasons, repeated procedures and investigations still unresolved, queue age, predicted versus actual completion, overdue mandatory work, setup changes, missed deadlines, stale calendars and schedule churn. Each can expose a workflow problem before complete findings arrive.

A changed score distribution does not prove accuracy decay, and fewer routine positives may just mean less testing. Compare completed audits as labels arrive. Record the expected audit cohort and pending completions so delayed or missing audits do not vanish from the denominator. The brief's full battery is the reference source for complete mechanism findings. [Helion brief](../problem-statement/helion_semiconductor_client_brief.md)

Sketch a **retraining trigger** *and* one operational **alert** for your system: the specific, measurable condition that fires each, who or what responds, and the risk of setting either threshold too tight or too loose.

**Helion answer: concrete proposed triggers.**

**Operational trigger:** when a scheduled nightly job reports failure, or has no validated completion by the agreed shift handover, suppress new ML recommendations and show SOP fallback. Fab IT owns recovery with the data owner. The exact clock time must match the MES schedule, not an invented 24-hour age limit on immutable measurements.

**Scheduling trigger:** replan uncommitted work after a new result, arrival, resource outage, duration overrun or urgency change. Validate resource state before offering slots. Escalate infeasible deadlines or maximum waits rather than dropping the job. Replanning changes a schedule, not the trained model.

**Retraining gate:** initiate a reviewed candidate-training request when the quality owner confirms model-related drift or a qualified process change **and** approves a new versioned audited dataset, split and evaluation protocol. An alert alone never retrains or promotes a model. New unqualified tools/materials route affected cases to SOP pending review. An audit-confirmed missed co-fault plausibly linked to the recommendation suspends the affected policy immediately for quality review.

Over-sensitive triggers consume scarce engineering time; permissive triggers prolong unsupported use. These are design defaults, not existing Helion operating rules. [Monitoring and owners](#helion-operations).

# **9 Putting It Together: Designing a Good ML System**

You now have the whole loop and a quality lens for each stage. The final skill is **holding it all at once**, because the stages are not independent. A decision in framing constrains your data; your data constrains your models; your serving latency budget constrains your features; your monitoring needs shape what you must log during training.

### **Design is a series of trade-offs**

There is no universally "best" ML system, only one that best fits *this* problem's constraints. Good design is the ability to make, and justify, choices like:

- A simpler, interpretable model vs a more accurate black box, when a human has to trust and act on the output.
- Batch scoring (cheap, simple, stale) vs real-time serving (fresh, complex, costly).
- Spending your effort budget on better data vs better models, usually the former wins, but not always.
- More configurability and abstraction vs a simpler system that's easier to understand.

### **A design checklist to carry into the E2E Pipeline**

Before you build, be able to answer:

- [ ] **Problem** - What decision does this inform? What's the ML task, the target, the metric, the baseline?
- [ ] **Data** - Where does it come from, how is it validated and versioned, and how do you avoid train/serve skew?
- [ ] **Training** - Is it config-driven, modular, reproducible, and tracked?
- [ ] **Evaluation** - Does the metric reflect the real cost of error? Have you guarded against leakage and checked important slices?
- [ ] **Deployment** - Which serving pattern, under what latency/cost/interpretability constraints, in what environment?
- [ ] **Monitoring** - What do you watch across the operational/data/model layers, and what triggers a retrain?
- [ ] **The loop** - How does what you learn in production feed back into the next iteration?

Pick **one** design decision from this notebook and trace its ripple effects through *at least three* other lifecycle stages. (For example: "we need real-time serving with a 100ms budget"; what does that force on features, models, data, and monitoring?)

**Helion answer: separate input cadences and follow their consequences.**

Choosing nightly model scores limits which cases have a qualified initial snapshot. Choosing rolling-horizon scheduling additionally requires current local calendars, reservations and procedure outcomes. These are distinct data contracts; the latter cannot be inferred from an overnight manufacturing table.

This serving choice changes evaluation: compare dispatch strategies on shared-resource shift/time blocks and account for backlog carryover, while retaining lot/time separation for classifier evaluation. It changes monitoring: stale resource feeds, violated maximum waits and schedule churn need their own alerts. It changes reproducibility: record each planning snapshot and commitment. It changes release design: model, diagnostic policy and scheduler configuration have separate versions and qualification checks. [Architecture](#helion-blueprint), [scheduler contract](#helion-scheduling).

Reread the qualities in Section 1. For a problem you care about, **rank the top three** for that specific problem and justify why those beat the others *here*. Then name the one you're consciously choosing to under-invest in, and the risk you're accepting by doing so.

**Helion answer: rank engineering qualities against consequences.**

1. **Testability and validation:** detect forbidden features, unresolved states incorrectly marked negative and invalid procedure transitions.
2. **Reliability:** keep investigation state and a working SOP fallback throughout night shifts.
3. **Observability:** make missing data, overrides, audit delays and uncertain outcomes visible to the four-engineer ownership team.

We under-invest in scaling to very high traffic and in automatic model promotion. The accepted risk is revisiting scheduling and staffing if demand grows substantially. The revised design uses CPU batch scoring, a bounded deterministic scheduler and local records. We avoid reinforcement learning and optimisation over every possible future diagnostic path. Scheduling complexity must earn its maintenance cost. The brief describes around 900 rejected cases per quarter, nightly inputs and qualification requirements. [Helion brief](../problem-statement/helion_semiconductor_client_brief.md)

# **10 From Understanding to Building**

This is where it all comes together. The main thing this notebook has been building is **judgement** - the ability to look at a problem and reason your way to a *good system*. Your headline output is that reasoning: a design you can lay out, justify decision by decision, and defend. **That design is the deliverable.**

An **optional** next step is to get **hands-on building it**, with an LLM assistant doing much of the actual typing - a **stretch goal** (see §10.3), not something to stress about delivering. Two reasons it's worth attempting *even though the assistant writes the code*:

- When an assistant can generate code on demand, the scarce skill is no longer *writing* it - it's **knowing what to ask for and why, and being able to tell whether what comes back is any good.** That is exactly the judgement you've practised.
- Actually wiring a pipeline together - data → train → evaluate → produce predictions, config-driven and reproducible - is a **rep worth having** before you're doing it for real. The aim of any build is a *clean, working, well-structured* pipeline - **not** a high score, and **not** a realistic dataset. A small or synthetic dataset is completely fine; the thinking about data quality, cost of error, and the rest lives in your *design*, not your build.

**A word on scope.** You **design** the whole lifecycle - including how you'd serve and monitor it - and that design is what you present and defend. You'll give a **short presentation** of it (expected), but the **oral defence - the live Q&A - is what you're actually assessed on**: reasoning held up under questioning. Present it clearly all the same - a clean walkthrough makes the defence easier for everyone. **Building** the pipeline (ingest → train → evaluate → predictions) is an **optional stretch** - encouraged reps for the project phase, never required and never penalised if absent. A pipeline that *runs end-to-end as intended* earns a **small bonus, on the order of 1-2% of the grade** - deliberately small, because a solid design beats a rushed build. **Deploying** it as a running service is a **further bonus** beyond that - if you want to take it that far, the companion **deployment reference notebook** walks you through the thinking (your LLM writes the code).

Everything you need is in this notebook. The three parts below bridge understanding and building: **what** an E2E pipeline is as a thing you assemble (10.1), **how** to turn your reasoning into a design blueprint (10.2), and **how** to drive an LLM to build it - if you take the stretch - without handing over the judgement (10.3).

### **10.1 The anatomy of an E2E pipeline**

You've met each stage on its own. Here is how they compose into one system you can actually build. Think of a pipeline as a **sequence of modular steps, wired together by configuration and run by an orchestrator** - not one long top-to-bottom script.

| Component | What it does | The artifact it produces | Qualities it embodies |
|---|---|---|---|
| **Data ingestion + validation** | Pull raw data; check schema, ranges, nulls; fail *loudly* on violation | A validated, versioned dataset (a Bronze → Silver → Gold progression) | Testability, reproducibility, observability |
| **Preprocessing + feature engineering** | Clean, transform, and build features - with the *same* logic used at training and serving | Feature definitions reused by both paths (no train/serve skew) | Modularity, reproducibility |
| **Training** | Fit the model from a *config* (model, features, split, seed, hyperparameters) | A trained model artifact + a logged experiment run | Configurability, reproducibility |
| **Evaluation + selection** | Score against a metric that reflects the cost of error; beat a baseline; check slices and leakage | A decision: promote this model, or don't | Testability, honest evaluation |
| **Serving** | Expose predictions in the pattern the use case dictates (batch / API / streaming / edge), in a pinned environment | A deployable service (often a container) | Reliability, scalability |
| **Monitoring + feedback** | Watch the operational / data / model layers; trigger a retrain when a threshold trips | Alerts, dashboards, and a loop back to training | Observability, reliability |

Two threads run through all of them:

- **Configuration** is the seam that makes the system *configurable and reproducible* - *what* model, *which* features, *which* thresholds live in a config file, not buried in code.
- **Orchestration** is what makes it *automated* - the steps run in order, on a schedule or a trigger, without anyone running each one by hand.

You do **not** need every box at full strength. A batch-scored internal tool may skip the feature store and the real-time API entirely; a high-stakes real-time system may invest heavily in both. *Which boxes you build big and which you keep lean* is the trade-off reasoning from this notebook, applied to your scenario - and that is exactly what the blueprint below forces you to decide.

Those components compose into a picture. Here is **one** illustrative shape - a generic risk-triage pipeline, *not* your case study. Every ML system shares the same training **spine**:

<p align="center"><img src="images/pipeline-spine.png" alt="The training spine: sources -> ingest + validate -> preprocess + features -> train -> evaluate + select -> model registry" width="100%" style="max-width:820px;"></p>

The interesting design lives in *how you serve that model and loop back*. Shape A scores in nightly batches:

<p align="center"><img src="images/pipeline-batch.png" alt="Batch serving: model registry -> scheduled scoring job -> predictions table -> a human acts on each -> monitoring (data / pipeline / model), with a retrain loop back to the registry" width="100%" style="max-width:340px;"></p>

Now flip **one** constraint - the decision is needed the instant a case appears, not overnight - and only the serving box changes:

<p align="center"><img src="images/pipeline-realtime.png" alt="Real-time serving: model registry -> prediction API (in a container) -> response to the caller -> a human acts on it -> monitoring (latency / data / model), with a retrain loop back to the registry" width="100%" style="max-width:320px;"></p>

Read these as **one shape for one set of constraints, not a template to copy.** The whole point of the notebook is that *your* diagram should look different because *your* constraints differ - a lighter monitoring layer if labels come fast, no real-time path if a nightly batch is fine, an extra human-review tier if the cost of a wrong call is high. And note the fidelity: rough boxes and arrows is exactly right. A sketch like this is what makes a system legible to someone else in minutes - which is why the blueprint below asks you to draw your own.

### **10.2 Turn your reasoning into a design blueprint**

Before any code, write a **one-page design blueprint** for your scenario. This is the single most useful artifact you can produce here: it is the **backbone of your presentation** *and* the specification you'll hand your LLM assistant to build from. Walk the lifecycle and commit to a decision - with a one-line *why* - for each line:

- [ ] **Problem** - the one-sentence statement: objective, ML task type, target, success metric, and the baseline to beat.
- [ ] **Cost of error** - which error is worse and roughly by how much, and the metric + operating threshold that follow from it.
- [ ] **Data** - sources, the validation checks you'll enforce, how you'll version it, and how you'll avoid train/serve skew.
- [ ] **Training** - how it's made config-driven, modular, reproducible, and tracked.
- [ ] **Evaluation** - the metric, the leakage guards, and the slices you'll check beyond the aggregate.
- [ ] **Serving** - the pattern (batch / API / streaming / edge) and *why*, plus how you'll pin the environment.
- [ ] **Monitoring** - what you watch across the operational / data / model layers, and the concrete retraining trigger.
- [ ] **Deliberately lean** - the one or two components you're *choosing* to keep simple, and what you bought with the complexity you saved.
- [ ] **Architecture sketch** - one boxes-and-arrows diagram of *your* system: the §10.1 components wired for the decisions above (like the two examples in §10.1, but yours). It should make your batch-vs-real-time choice, your human-review tier, and your feedback loop visible at a glance. Hand-drawn, mermaid, or ASCII is all fine - it's read for what it shows about your thinking, not how polished it looks.

Fill this in for your case study in the cell below - it is what you'll present *and* what you'll build from. The written decisions and the sketch are two views of the same design: the sketch is what you'll put on screen and talk over; the decisions are the *why* behind each box.

> **The blueprint is for you; the pre-read is what you hand in.** This blueprint is the backbone of your presentation and (if you build) your spec - keep it as full as you like. What you *submit* the day before your viva is something lighter: a short **pre-read** (under 500 words, plus your architecture sketch) that picks out the **most interesting decisions and trade-offs** from this blueprint - the things you most want your mentors to know. Assemble it with **`PRE_READ_TEMPLATE.md`**, and see **`PRESENTATION_RUBRIC_APPRENTICE.md`** for exactly how the viva is assessed. Mentors read it to prepare *pointed* questions, so treat it as the map of your thinking that others will probe. It is **not graded at face value** - an LLM can write it; it can't sit the viva. The presentation sets your design up; what actually **scores is the oral defence** - how you hold it up under live questioning.

<a id="helion-blueprint"></a>
## Helion §10.2 design blueprint

**Integrated design proposal, revised 21 September 2026.** Scheduling broadens the original one-decision assignment at the user's request. No trained model, implemented scheduler, measured savings or external scope approval is asserted.

| Decision | Choice and why |
|---|---|
| Problem | Complete adequate investigations across rejected stacks at minimum expected operational cost, subject to completeness, resource and turnaround constraints. One seven-output model supports diagnostic selection; a deterministic scheduler allocates eligible work. |
| Cost of error | Missing co-faults harms completeness; unnecessary work, delays and poor changeovers consume resources. Use comparable non-overlapping costs. Fault weights rank eligible work but never override mandatory work, evidence preservation or maximum waits. |
| Data | Version initial MES snapshots, independent audited labels and current local resource/queue state separately. Preserve lot/time partitions, missingness meanings and diagnostic evidence provenance. Scheduling records are additional data requirements. |
| Training | Propose shared preprocessing and regularised logistic heads in one CPU model artifact. Fit on qualified complete audit history, recording configurations and artifacts. Scheduling is deterministic planning, not model retraining. |
| Evaluation | Compare incumbent/incumbent, new-selection/incumbent-dispatch, incumbent-selection/new-scheduling and both. Attribute cost/turnaround changes separately, using adequate audited completeness and shared-resource-aware prospective comparisons. |
| Serving | Nightly fault scores, event-updated diagnostic eligibility and local rolling-horizon scheduling. Protect commitments; humans confirm test/slot and closure. Invalid diagnostic data uses SOP, invalid resource state uses manual dispatch. |
| Monitoring | Watch data/job health, audited faults, resource freshness, open-case age, deadlines, setups and schedule churn. Replan on operational events; retrain only after the separate reviewed model gate. |
| Deliberately lean | One predictive candidate, transparent coverage/cost priority and bounded constraint-based scheduling. Avoid reinforcement learning, cloud dependencies and automatic promotion; report infeasibility rather than hiding it. |

Scenario authority: [brief](../problem-statement/helion_semiconductor_client_brief.md). Scheduling resources and benefits remain unsupported by the [supplied package](../README.md#scope-of-realism). The scope expansion originates in the user's request, not a revised client brief.

### Architecture

Offline viewing: [diagram image](images/helion-architecture.png) and [editable Mermaid source](images/helion-architecture.mmd).

```mermaid
flowchart TB
  subgraph N["Nightly fault scoring"]
    direction LR
    MES["MES snapshot"] --> VALID["Validation and<br/>shared features"] --> SCORES["Initial probabilities"]
  end
  subgraph D["Diagnostic selection"]
    direction LR
    STATE["Mechanism states<br/>and qualified findings"] --> POLICY["Coverage and cost<br/>priority policy"] --> OPTIONS["Eligible alternatives<br/>and mandatory work"]
    POLICY -->|"unsupported"| SOP["Existing diagnostic SOP"] --> OPTIONS
  end
  subgraph S["Scheduling across stacks"]
    direction LR
    RES["Current resources<br/>queues and commitments"] --> SCHED["Rolling-horizon<br/>constraint scheduler"] --> PEOPLE["Engineer and coordinator<br/>confirm test and slot"] --> EVENTS["Execution and findings<br/>actual time and cost"]
    SCHED -->|"infeasible or stale"| MANUAL["Manual dispatch<br/>and escalation"] --> PEOPLE
  end
  subgraph G["Offline qualification"]
    direction LR
    AUDIT["Independent 5%<br/>full-battery obligations"] --> LABELS["Complete audited labels"] --> TRAIN["Offline training<br/>and evaluation"] --> REVIEW["Qualified versioned<br/>release"]
  end
  G -->|"model release"| N
  N -->|"initial scores"| D
  N -->|"versioned features"| G
  D -->|"alternatives and constraints"| S
  S -->|"feasible slots, costs and new findings"| D
  G -->|"protected audit work"| S
  S -->|"plans, outcomes and backlog"| MON["Operational and outcome monitoring"]
  N -->|"batch health"| MON
  G -->|"audit outcomes"| MON
  MON -->|"reviewed candidate request"| G
```

Nightly scores feed per-stack diagnosis. Eligible alternatives and obligations feed the scheduler, which returns feasible slots/costs and later execution findings. Independent audit work is protected from ordinary priority suppression. Monitoring distinguishes operational replanning from reviewed model retraining. All components run inside the fab.

### Logical interfaces

**Diagnostic inputs:** qualified initial snapshot, seven timestamped initial probabilities, mechanism states/findings, procedure history, catalogue/SOP version, prerequisites and evidence-preservation conditions.

**Diagnostic outputs:** all justified eligible alternatives, covered unresolved mechanisms, mandatory work and closure obligations, initial weighted priorities, prerequisites and fallback reasons. Do not pass only the top-ranked test.

**Scheduling inputs:** those alternatives across stacks, current resource/staff calendars, queues, commitments, duration and setup estimates, batching/specimen compatibility, approved cost rates, deadlines, urgency and maximum waits.

**Scheduling outputs:** proposed chosen procedure/resource/start/end, assumed queue/setup/cost and planning timestamp/version, unchanged commitments, blocked reasons and escalation. The engineer confirms diagnostic suitability; the coordinator confirms dispatch. A reservation consumes only one chosen alternative. A locally logged result updates supported mechanism states and starts another planning cycle.

Neither component has authority to declare absence from a low score, close a case automatically, skip mandatory work or ship material. No API or runtime is implemented. See [diagnostic policy](#helion-policy) and [scheduling](#helion-scheduling).


### **10.3 Build it with an LLM - and keep the judgement yours**

> **This is a stretch goal - treat it that way.** The build is where you *practise* wiring a pipeline together: valuable reps for the project phase, but **not** the thing you're assessed on and **nothing to stress about delivering.** A working pipeline is worth a **small bonus (~1-2% of the grade)** on top of your viva - real credit, but nowhere near enough to justify borrowing time from your design. Your design and how you defend it come first. **A solid design beats a sloppy build of a half-baked one** - if the time is a choice between making your design watertight and rushing an implementation, make the design watertight. A clean, working pipeline is a nice bonus on top of good judgement, never a substitute for it.

You have a capable coding assistant; use it. But the division of labour is the whole point: **the design decisions in your blueprint are yours; the LLM is the implementer.** Left to its own devices it will readily produce plausible code that violates the very qualities you decided mattered - a single monolithic script when you wanted modules, statistics fit over the whole dataset before splitting (leakage), a parameter hard-coded where you meant to expose it in config. Your value is catching exactly that, because *you* know what "good" means for this problem and the model only knows what's common.

A few working habits:

- **Feed it the blueprint, not just a task.** *"Build me a churn model"* gets you a throwaway notebook. *"Build a config-driven training module with data-loading, preprocessing, and model steps as separate classes, reading the model choice and hyperparameters from a YAML config, with a fixed random seed"* gets you a system.
- **Specify structure and interfaces, not just behaviour** - `.py` modules over a notebook, clean seams between components, config over hard-coded values. These are decisions the LLM won't make well unless you make them.
- **Verify against your design, not just against "it runs."** Is it modular and config-driven? Is the run reproducible from config + seed? Does it validate incoming data and fail loudly? Does the metric it optimises match your cost-of-error?
- **Own the decisions the model can't make for you** - the metric, the operating threshold, the serving pattern, what to monitor and when to retrain. Those come from your scenario, not from any amount of generated code.

When the build is **practice** - a rep on a small or synthetic dataset - lean on the *structural* checks first (modular, config-driven, reproducible, runs end-to-end). The data-quality checks (leakage, train/serve skew) bite hardest on real data; here, knowing they belong on the list is itself the skill.

This is why the understanding was the point all along: the code is now cheap to produce, but only your judgement can tell whether it's the *right* code.

**If you take the build stretch, here's the shape.** You don't need to invent a project structure - structure it like a standard ML pipeline repo and have your LLM scaffold it:

- `src/` - your pipeline as **`.py` modules and classes**, not a notebook
- `run.sh` - one command that runs it end to end
- `requirements.txt` - pinned dependencies
- a **config** (YAML / env vars / CLI) driving model, features, and parameters
- a short `README.md`

This is the same shape the **AIAP Technical Assessment** you submitted asked for - recall its required structure and expectations.

Two notes for *this* build: your **data is LLM-synthesised** - generate it as your *first* step (a CSV or table is fine; no database needed) - and the build stops at **predictions** (ingest → train → evaluate → predictions). Serving and monitoring you *design*, not build. And remember the framing from §10: this is a **stretch**, so a solid design with a rough or unfinished build is completely fine - never sacrifice the design to rush the code.

**What "verify against your design" looks like in practice.** That bullet is easy to nod at and hard to do, because an assistant's *dangerous* bugs are not the ones that crash - those you catch on the first run. They are **silent**: the code runs end to end, prints a good score, and is quietly wrong. Fluent, confident code is engineered to survive a careful read, so you don't out-read it - you make it **prove itself** with checks that fail loudly. A handful of these recur in generated ML code, each with the check that actually catches it:

| Silent failure the assistant loves to introduce | What it costs you | The check that catches it |
|---|---|---|
| Fits a scaler / imputer / encoder on the **full data before splitting** | Test statistics leak into training; offline score inflates, production collapses | Assert the transformer was `fit` on train rows only; fit *inside* a `Pipeline` / CV fold |
| Uses a feature only knowable **after** the outcome | A "perfect" model that cannot exist at prediction time | Point-in-time check: every feature must be available at the prediction timestamp |
| `try: ... except: pass` around parsing / IO | Rows silently dropped or defaulted; your data quietly shrinks and skews | Assert row counts before/after each step; ban bare `except` in review / CI |
| `fillna(0)` / silent `astype` coercion | A missing value becomes a *real* zero, changing the feature's meaning | Assert null-rate and dtype against a schema (e.g. Pandera) before training |
| Feature computed one way in training, another in serving | Train/serve skew; eval looks fine, live is wrong | One shared feature function, unit-tested; diff train vs serve output on a fixture |
| Hard-coded fallback seed / hidden nondeterminism | "Reproducible" runs that quietly aren't | Run twice from the same config + seed; assert identical artifacts |

The move in every row is the same: turn a thing you would have to *notice* into a thing that *fails the build*. The demo below shows why noticing isn't enough.

*Runnable - the labels here are **pure noise**, so an honest model must score about 0.5. Watch a single line of leaked feature-selection manufacture a score well above it. Run it, then read both versions and notice that each one looks perfectly fine.*

In [ ]:
# Run me. y is RANDOM noise here, so an honest model MUST score ~0.5. Watch
# leakage fabricate a score above that - exactly what an assistant's
# "pick the best features first, then split" code silently does.
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

rng = np.random.default_rng(0)
n, p = 400, 3000                       # few rows, MANY mostly-noise features
X = rng.normal(size=(n, p))
y = rng.integers(0, 2, size=n)         # label is UNRELATED to X - pure noise

Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.3, random_state=0)

def top_k_by_corr(features, target, k=20):
    # pick the k features most correlated with the target
    corr = np.abs([np.corrcoef(features[:, j], target)[0, 1] for j in range(features.shape[1])])
    return np.argsort(corr)[-k:]

with np.errstate(divide="ignore", over="ignore", invalid="ignore"):  # mute a cosmetic numpy/BLAS matmul warning
    # WRONG: choose features using ALL the data (train + test) - the classic leak.
    sel_leak = top_k_by_corr(X, y)
    m = LogisticRegression(max_iter=1000).fit(Xtr[:, sel_leak], ytr)
    acc_leak = accuracy_score(yte, m.predict(Xte[:, sel_leak]))

    # RIGHT: choose features from TRAIN ONLY, then judge on the untouched test set.
    sel_ok = top_k_by_corr(Xtr, ytr)
    m = LogisticRegression(max_iter=1000).fit(Xtr[:, sel_ok], ytr)
    acc_ok = accuracy_score(yte, m.predict(Xte[:, sel_ok]))

print(f"leaked feature-selection -> reported accuracy: {acc_leak:.3f}")
print(f"clean  feature-selection -> honest   accuracy: {acc_ok:.3f}   (~0.5, as it must be)")
print()
print("The leaked number is the one that looks great in the notebook and the one")
print("that evaporates in production. Reading the code would not save you here -")
print("both versions read fine. A test asserting selection saw TRAIN ONLY would.")


Take **one** component from your blueprint (say, data validation, or the training config). Write the instruction you would give an LLM to build it - specific enough that the result would satisfy *your* design, not just any generic version. Then list the **two things you'd check in its output** to confirm it actually did what you asked.

**Design exercise only: an instruction for a future implementation.**

“Build a validation module for the rejected-stack feature snapshot. Accept an explicit feature allowlist, schema and split manifest. Validate unique stack IDs, one lot per stack, required acceptance observations and feature availability at the decision cut-off. Preserve permitted missing metrology with its reason and measured-die count. Reject any `fault_*`, `p_*`, post-investigation annotation, reliability outcome or `timing_margin_ps` predictor. Return a validation result with reasons; never silently drop rows or coerce unknown findings to zero.”

Two checks: inject `defect_type` and require rejection; provide a valid unsampled-metrology fixture and require acceptance with preserved missingness. This is a prompt and verification design, not an implemented module.

Pick one silent failure from the table above that is realistic for your scenario. Sketch the automated test (prose or pseudo-code) that would catch it before it reached production, and explain why a careful human code review would probably have *missed* it.

**Design exercise only: catch an unknown label silently becoming zero.**

```text
Given: no procedure assessing die crack, and no complete-battery finding
When: a diagnostic record is normalised
Then: fault_die_crack remains unknown, not 0
      its observed-label mask is false
      it cannot count as a true negative in evaluation
      its mechanism state remains untested

Given: a qualified negative diagnostic finding for die crack
Then: the state may become confirmed_absent with evidence provenance
```

A generic `fillna(0)` can look like ordinary preprocessing during code review. The fixture exposes its different meaning for a diagnostic target. Masking unknowns is necessary for correctness, but does not remove the bias of selectively choosing which tests to run. Our initial operational fitting/evaluation reference therefore uses independent complete audits. [Data design](#helion-data).

Data pipelines, model pipelines, deployment, monitoring - the DataOps notebook laid the data foundation, and here you've seen how the rest of the system clips onto it into a single loop. The skill that carries forward isn't any one tool; it's the judgement to look at a new problem and reason your way to a system that *fits* it.

That judgement is exactly what your mentors during the project phase will ask of you.

<h1><center>End of Notebook - Designing a Good ML System</center></h1>

<a id="helion-evidence"></a>
## Appendix A. Evidence and unresolved questions

**Verified-in-files audit, 18 September 2026:** filter `ml_stack_labels.final_test_fail == 1`, join the resulting IDs to the feature and truth tables by `stack_id`, count partitions and positive labels, and inspect all 19 CSV headers. This is a read-only data audit, not model evaluation. Sources: [acceptance labels](../data/ml/ml_stack_labels.csv), [feature snapshot](../data/ml/ml_stack_features.csv), [synthetic stack truth](../data/simulation_truth/simulation_truth_stack.csv), [dictionary](../data/data_dictionary.csv).

| Claim | Status and source | Consequence |
|---|---|---|
| Around 900 rejects/quarter; 1 in 20 receives the complete battery | Brief-stated: [business context](../problem-statement/helion_semiconductor_client_brief.md#business-context) | About 45 complete cases/quarter is a planning expectation, not this CSV's audit count |
| 17,793 assembled stacks; 916 rejected | Verified from acceptance labels; [documented cohort](../README.md#rejected-stack-cohort-and-seven-targets) | Filter rejects before the diagnostic task |
| 653/126/137 rejected train/validation/test cases | Verified join to feature split; [partitions](../README.md#rejected-stack-cohort-and-seven-targets) | Preserve lot/time partitions rather than random row splitting |
| Seven complete synthetic mechanism indicators; 21 multi-fault cases | Verified from truth restricted to reject IDs; [label semantics](../README.md#rejected-stack-cohort-and-seven-targets) | These do not reproduce selectively observed real findings |
| Actual thickness field is `final_die_thickness_um`, complete on all 192,000 dies | Verified in [die metrology](../data/manufacturing/die_metrology.csv); exact `die_thickness_um` is absent | The brief's sampled-thickness wording is not an exact description of this CSV |
| No observed detailed metrology on 1,214 stacks, including 57 rejects | Verified from [feature snapshot](../data/ml/ml_stack_features.csv): both sampled aggregates blank exactly when observed-die count is zero | Preserve empty aggregate and coverage count; do not fabricate a measured zero |
| `procedures_run`, `diagnostic_hours`, `full_battery_sample` absent | Verified header scan across all CSVs; [workflow limits](../README.md#where-diagnostic-test-savings-could-come-from) | No observed procedure savings or audit-only evaluation can be computed here |
| New bonder and new underfill supplier coming | Brief-stated: [business context](../problem-statement/helion_semiconductor_client_brief.md#business-context) | Prepare qualification and monitoring triggers; do not invent changed failure rates |
| A useful next-test ranking will save effort | Hypothesis, explicitly unproven; [value boundary](../README.md#where-diagnostic-test-savings-could-come-from) | Test the whole workflow, including fallback and repeat work |
| Diagnostic resource calendars, queues, setups, batching, staffing and deadlines | Not supplied as a qualified diagnostic-lab operational dataset; [simulation boundaries](../README.md#scope-of-realism) | Collect these before scheduling validation; assembly tool IDs are not diagnostic equipment availability |
| Catalogue sensitivity, SOP thresholds, complete-enough criteria and rates | Unknown: brief provides a high-level catalogue, not a qualified SOP | Required client decisions before live use |

| Synthetic target | Positive rejected stacks |
|---|---:|
| `fault_dram_electrical` | 97 |
| `fault_tsv_open_short` | 108 |
| `fault_microbump_open_bridge` | 137 |
| `fault_die_crack` | 47 |
| `fault_warpage` | 247 |
| `fault_underfill_void` | 206 |
| `fault_delamination` | 96 |

Counts overlap because mechanisms coexist. The 938 positive labels across 916 rejected cases include 21 multi-fault stacks; these are different units. [Synthetic reference labels](../data/simulation_truth/simulation_truth_stack.csv), selected by [rejected-stack IDs](../data/ml/ml_stack_labels.csv).

**Scope update, 21 September 2026:** the user requested an integrated diagnostic-selection and scheduling design. This changes the proposed objective and interfaces, not the historical CSV facts above or the original brief.

**Decision register:** every design decision has an owner, rationale, alternative and review trigger in the [viva guide](viva_preparation.md). Unknown client inputs remain labelled deployment prerequisites, not missing answers to the course exercise.

**Additional synthetic inputs, 21 September 2026:** [SYN-OPS-001](../synthetic-data-assumptions/operation-assumptions/synthetic_operating_assumptions.md) supplies invented resource costs, staff/equipment calendars and scoped outcome parameters. [MOCK-ENG-001](mock_engineering_inspection_rules.md) supplies an invented comparator and all-seven closure rule. Neither resolves the absence of real diagnostic histories, rates, dispatch or qualified SOP. See [Appendix I](#helion-workflow-review).


<a id="helion-policy"></a>
## Appendix B. From probabilities to a next procedure

### Candidate model

**Proposed:** one multi-label logistic model artifact contains shared preprocessing and seven regularised binary output heads. Each estimates an initial mechanism probability from the same available snapshot. Outputs need not sum to one. This remains one versioned diagnostic-support model. The broader system supports two coordinated operational decisions, procedure eligibility and resource/time allocation, rather than seven separately deployed fault services. Logistic classification uses a sigmoid link with regularisation; see the [official linear-model documentation](https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression).

The choice favours a small parameter set and reviewable contributions given limited labels. It may miss nonlinear relationships and does not model joint fault probabilities. A high standalone score does not establish extra information beyond the rules. Feature families adding manufacturing context must demonstrate incremental workflow value over the same-input inspection baseline. No candidate is fitted here.

### Evidence state and allowed transitions

| State | Meaning | Permitted transition |
|---|---|---|
| `untested` | No qualified finding for this mechanism | Appropriate procedure result may establish presence/absence or be inconclusive |
| `confirmed_present` | Qualified finding establishes the mechanism | Retain evidence; correcting it requires documented expert review |
| `confirmed_absent` | Qualified procedure evidence excludes the mechanism within its validated scope | Retain coverage/detection-limit evidence; contradictory findings require review |
| `inconclusive` | A procedure did not settle the mechanism | Keep unresolved; use SOP repeat/alternative/escalation |

Initial probabilities and package inspection values do not automatically change these states. A finding applies only to the mechanisms and detection limits that procedure actually supports. A confirmed TSV open leaves microbump status unchanged. These proposed controls address the brief's [coexisting-fault failure mode](../problem-statement/helion_semiconductor_client_brief.md#business-context).

### Catalogue and ranking

The following values come from the [brief's procedure table](../problem-statement/helion_semiconductor_client_brief.md#business-context). Coverage is high-level; operational exclusion claims require the lab's validated scope.

| Procedure | Stated role | Stated duration | Inconclusive rate | Policy treatment |
|---|---|---|---:|---|
| X-ray/CT | Warpage, underfill void, gross delamination | 45 min | 10% | Rank if it can resolve an outstanding question |
| Acoustic microscopy | Delamination, void | 1.5 h | 15% | Rank with qualified scope; do not assume every delamination is gross |
| Electrical fault isolation | DRAM/base electrical, TSV open/short | 4 h | 20% | Rank for outstanding electrical questions |
| Cross-section/SEM | Microbump, die crack, TSV | 2 days, destructive | 5% | Eligible only after preservation/prerequisites and engineer authorisation; necessary expensive work receives explicit scheduling obligations |
| Thermal/IR | Electrical fault localisation | 1 h | 30% | Localisation/preparation when SOP requires it; no independent exclusion of a fault |

**Proposed weighted fault-coverage priority:** for stack `i`, eligible procedure `t` and candidate slot `s`, use:

```text
R_i(t, s) = (1 - q_t) * sum(w_m * p_im for m in U_i intersect C_t)
            / c_it(s)
```

`U_i` includes both untested and inconclusive mechanisms. `C_t` is the procedure's validated coverage, `p_im` is an initial model probability, and `w_m` is an approved/versioned fault-importance weight. Equal weights are only an illustrative starting assumption. `c_it(s)` is a positive, comparable incremental cost estimate for that procedure/slot. Queue and setup costs depend on the proposed schedule and preceding work, so they are not fixed test attributes.

The numerator rewards coverage of plausible positive mechanisms, not the probability of any fault, complete expected uncertainty reduction or the full value of negative findings. The factor `1-q_t` is a simplifying inconclusive adjustment, not sensitivity or specificity. Results may settle some covered questions but leave others inconclusive; update only the supported mechanism states. No global optimality follows from this ratio.

Use one agreed currency or explicitly converted cost scale. Price non-overlapping engineer effort, equipment occupancy, setup/cleanup and consumables. Do not add hours, dollars and a destruction flag directly, or count procedure duration again when its labour/machine use is already priced. Default queue-delay treatment is an explicit turnaround/maximum-wait constraint and a separately reported measure; include a monetary delay penalty only if its meaning and rate are approved. Destruction and evidence preservation are primarily feasibility constraints. A high score cannot buy permission to destroy needed evidence.

Apply mandatory SOP steps and dependencies before ranking. Qualified gross-delamination scope is necessary before X-ray receives delamination credit; IR localisation alone cannot exclude a mechanism. Required expensive procedures remain obligations even with low scores. Once destructive prerequisites and authorisation are satisfied, the scheduler must allocate necessary work or escalate blocked capacity; cost cannot make it disappear indefinitely. Missing cost data means no numerical cost ranking for that option, followed by the qualified manual route, not exclusion of necessary diagnosis.

The diagnostic policy sends eligible alternatives to the scheduler. The scheduler evaluates feasible slots, queue delays and sequence-dependent setups, then returns costs and a proposed choice. Compare/revise within a bounded planning run and confirm one option. This avoids assuming a fixed ranking is independent of the schedule it helps create. Ties follow incumbent diagnostic order unless approved urgency/resource constraints require another eligible option.

After each result, change the supported states, recompute eligibility and replan future work. Preserve other initial probabilities and label them as initial estimates. The system does not invent Bayesian posterior updates. Inconclusive findings remain unresolved; repeat/alternative paths follow SOP and include the effort already consumed. See [scheduling](#helion-scheduling).

### Closure and where savings could arise

The policy never emits an autonomous STOP. The engineer applies the existing qualified completeness standard, records supporting evidence and documents any unresolved item the standard permits. The exact standard and acceptable unresolved states are not supplied and must be obtained before live use.

Diagnostic-selection benefit requires justified changes to optional work, repeats or handling without weakening completeness. Scheduling can additionally reduce queue delays or sequence-dependent setup costs even if the same tests run. Reordering does not reduce the fixed execution duration of those tests. Report these outcomes separately, including any cost increase accepted to meet an approved deadline, and do not attribute scheduler-only benefit to the classifier. [Brief's value requirement](../problem-statement/helion_semiconductor_client_brief.md#constraints--scope).

**Costed paper exercise:** [Appendix I](#helion-workflow-review) evaluates this heuristic with SYN-OPS-001 standard resource costs, which value consumed capacity and are not synonymous with avoidable cash. Its q discount does not incorporate the separately invented sensitivity/specificity or value all negative evidence. Expected remaining investigation cost is the broader decision objective; a complete continuation model and validated planner are not delivered here. The current seven-output classifier remains an initial fault predictor.


<a id="helion-data"></a>
## Appendix C. Data and reproducibility design

### Two evidence tracks

**Supplied synthetic track:** use rejected IDs from `data/ml/ml_stack_labels.csv`; join the feature snapshot, allowed assembly context and allowed initial acceptance observations one-to-one by stack ID; extract only the key and seven `fault_*` columns into a separate target view. Verify row counts and unmatched/duplicate keys before accepting joins. `stack_training.csv` and `metadata/ml_task.json` remain the old pre-acceptance binary benchmark. [Task boundary and proposed joins](../README.md#information-available-at-diagnostic-selection).

**Proposed operational track:** initially fit/evaluate only independently selected complete-audit cases, pooling sufficient history and preserving time/lot separation. Selectively observed routine findings support descriptive monitoring and comparison of audit selection, not implicit negative labels. Masking unknown entries would avoid false zeros but would not remove selection bias. Synthetic truth is not a substitute for real audit labels. Audit membership must be assigned independently of scores, including cases that follow SOP fallback. [Brief's source of complete findings](../problem-statement/helion_semiconductor_client_brief.md#business-context).

### Feature boundary

| Use | Treatment |
|---|---|
| Manufacturing/package snapshot and sampled-die aggregates | Candidate predictors with availability and unit checks |
| Initial acceptance measurements | Eligible at this decision checkpoint; audit each field and simulated generation shortcut |
| IDs, split labels, timestamps, audit flag | Join/split/provenance metadata, not model predictors |
| `defect_type`, `defect_stage`, `defect_severity`, `final_disposition` | Exclude: investigation/outcome annotations |
| `fault_*`, simulator `p_*`, latent states, later reliability | Exclude from predictors; isolate target columns only |
| `timing_margin_ps` | Exclude initially because of outcome-derived simulated sign |

The file can mix legitimate measurements and prohibited annotations, so whole-table imports are inappropriate. The old feature timestamp precedes acceptance and does not prove later diagnostic chronology. [Feature and chronology limits](../README.md#information-available-at-diagnostic-selection).

### Missingness and units

Read the reason flag alongside missing detailed measurements: `not_sampled` and `instrument_dropout` describe different events. In the supplied 192,000 die records, these reason counts are 144,486 and 982; 46,532 have `none`. `metrology_sampled=1` includes subsequent dropouts, whereas `n_dies_with_detailed_metrology` counts successful observations. The final thickness field is complete and named `final_die_thickness_um`; the exact brief field `die_thickness_um` is absent. [Die metrology](../data/manufacturing/die_metrology.csv), [feature snapshot](../data/ml/ml_stack_features.csv), [dictionary](../data/data_dictionary.csv).

Stage-specific process fields may be structurally inapplicable. Preserve a true measured zero. Never infer a missingness reason from a blank value alone. A proposed applicability indicator comes from the process stage; there is no claim that a literal `not_applicable` flag exists in the CSV. Validate within stage and avoid pooling meaningless absent-stage measurements into an aggregate. [Process-field definitions](../data/data_dictionary.csv).

Proposed preprocessing fits numeric medians, scales and categorical encodings on training data only. Retain missingness indicators and observed-metrology counts. Apply missing-measurement imputation only within the applicable stage or valid stack feature; structural absence carries applicability information and must not receive an ordinary global median as though an instrument reading were lost. Where a feature has no training observations, exclude it with a logged reason rather than invent a median. Missing required acceptance data, unsupported tool/material conditions or a schema failure routes the case to SOP; expected sampled-metrology gaps alone do not. [Leakage and preprocessing guidance](https://scikit-learn.org/stable/common_pitfalls.html#data-leakage).

Keep the mean per-via copper-fill void-volume percentage surrogate, underfill void area as a percentage of inspected underfill cross-section area, and separated area as a percentage of inspected package interface area separate. Their different denominators prohibit treating them as interchangeable or adding them into a physical total. [Dictionary field descriptions](../data/data_dictionary.csv), rows keyed by `tsv_inspection.tsv_copper_void_pct`, `stack_assembly.underfill_void_pct` and `stack_assembly.delamination_area_pct`.

### Splits and provenance

Retain the supplied ordered lot partitions and configured 21-day embargo. The verified lot-release gap at each boundary is 516 hours: 21 days plus the normal 12-hour lot spacing. Stack-level IDs match one-to-one across the feature, label, assembly, test and truth files; genealogy in `stack_membership` is one-to-many and must be aggregated before joining predictors. [Lot records](../data/manufacturing/lots.csv), [membership](../data/assembly/stack_membership.csv), [split specification](../metadata/ml_task.json).

Fit every transformation and choose model/policy settings within training/validation. Existing test data has already been examined in earlier audits, so further results would be retrospective. New claims require untouched later cases. [Evaluation limits](../README.md#proposed-evaluation-and-monitoring).

For operational records, add event and availability timestamps for initial results, procedure selection, procedure completion, confirmed findings and closure. Save catalogue/SOP/model/policy versions, audit selection and completion, unknown versus confirmed-negative status, overrides, engineer effort, equipment occupancy and elapsed turnaround separately. These fields are a proposed collection contract, not data currently present.

Proposed future training uses a small regularisation search on lot/time-respecting training folds, with final settings frozen before holdout evaluation. Preserve positive and negative support per head; a missing class or inadequate evidence prevents qualification for that scope. Assess calibration instead of assuming logistic outputs are calibrated. Store code/data hashes, configuration, environment and fitted preprocessing/weights locally. No training or search is executed for this submission.


### Additional operational data for scheduling

The existing `assembly_tool_id` identifies simulated assembly context, not a diagnostic resource inventory or live queue. The simulation explicitly excludes dispatching, queueing, resource capacity and costs. [Scope of realism](../README.md#scope-of-realism).

Proposed collection contract: diagnostic resource IDs/capabilities; maintenance and availability calendars with timestamps; staff skills and attendance windows; processing, setup and cleanup durations and resource demands; sequence-dependent setup matrices; batching rules and sample compatibility; arrival, reservation, actual start/end and release events; deadlines, urgency, maximum waits; procedure dependencies and destructive evidence prerequisites; independent audit assignments; rates and consumables; current plan, commitments and override reasons. Distinguish forecasts from realised durations/costs and retain each planning snapshot.

Equipment occupancy and hands-on staff attendance need separate intervals. A two-day procedure does not imply two days of continuous engineer attendance. Derive queue delay from the candidate schedule and accepted work; do not treat a historic queue average as current capacity. These records require a qualified local update path within the fab in addition to nightly MES consolidation. None is claimed to exist in the supplied files. No new metrology sampling is requested.


<a id="helion-evaluation"></a>
## Appendix D. Evaluation, economics and evidence gates

### Objective, measurements and constraints

**Business objective:** minimise expected total operational cost of completing diagnostically adequate investigations across a defined cohort, subject to diagnostic completeness, safety/evidence preservation, resource feasibility and approved turnaround/maximum-wait requirements. Compare total cohort cost and cost per eligible rejected stack over a common prespecified follow-up. Until rates and cost boundaries are agreed, report time/resource components separately rather than claim total-cost improvement.

Record hands-on engineer hours, equipment occupancy, setup/cleanup time, consumables and any separately justified delay charges without overlap. Report mean and p95 turnaround, deadline misses, queue/open-case age, repeats and schedule changes alongside cost. High utilisation alone is not success. Keep pending investigations, accumulated effort and remaining obligations visible; deferring expensive work beyond the horizon or observation window is not a saving. Lower effort-to-date with more unfinished cases cannot establish improvement.

Completeness guardrails include audited missed mechanisms, missed co-faults, erroneous absence findings and unresolved investigations, with explicit denominators and per-fault support. Initial fault recall/AP/calibration remain supporting model metrics. Queue performance cannot compensate for a diagnostic-completeness violation. Numerical completeness margins, deadlines, costs and statistical evidence thresholds remain client inputs.

### Separate selection and scheduling benefits

| Comparison condition | Diagnostic selection | Dispatch/scheduling |
|---|---|---|
| A: incumbent baseline | Existing engineering rules | Actual current dispatch practice, to be documented |
| B: selection change | Proposed model-assisted policy | Same current dispatch as A |
| C: scheduling change | Same rules as A | Proposed rolling-horizon scheduler |
| D: integrated change | Proposed model-assisted policy | Proposed rolling-horizon scheduler |

Compare B with A for selection, C with A for scheduling, and D with A for combined value. Inspect whether the combination changes either component's benefit. Hold workload, available resources, procedure coverage, cost definitions and completeness constraints comparable. Do not assume current dispatch is FIFO without evidence. Scheduling improvements alone do not demonstrate a need for ML. The full battery provides reference findings and remains mandatory for selected audit cases; it is not the usual-cost comparator. [Brief baseline](../problem-statement/helion_semiconductor_client_brief.md#business-context).

### Staged evidence

1. **Supplied files:** establish cohort, missingness and leakage limits. They contain no validated diagnostic schedule or cost history. Any future simulated routes, resource calendars or outcomes are declared assumptions and cannot establish actual savings. [Package boundary](../README.md#scope-of-realism).
2. **Qualified shadow study:** log alternative procedure/slot proposals while incumbent decisions continue. Check feasibility, data freshness, inferred delays, conflicts and workload effects. Replay cannot prove counterfactual hours/cost saved, and q alone cannot generate realistic conditional diagnostic outcomes.
3. **Approved prospective comparison:** select adequately powered contrasts from A–D rather than insisting on an underpowered four-arm pilot. Compare shared-resource lab shifts/time blocks with controlled backlog carryover, resource availability and process changes; independent stack randomisation cannot be analysed as if cases had separate queues. Retain lot/time separation for classifier evaluation and account for lots and shared-resource blocks in uncertainty estimates. Pre-register endpoint definitions, follow-up and completeness limits.

Independent audits apply across recommendation, scheduling and fallback groups. Protect their lab capacity and log selected, pending and completed cases so throughput optimisation cannot suppress ground truth. Freeze evaluated findings/effort before reference findings become available, and qualify destructive sequencing so reference evidence survives. Account consistently for common audit work and any incremental audit burden.

Promotion requires adequate diagnostic evidence, feasible service obligations, net value after both model and scheduler overhead, and software qualification. Wide uncertainty or an infeasible hard constraint means hold or escalate, not silently loosen the standard. No optimality, sample-size adequacy or improved performance is established here.

### Small audit cohort

From the brief, 900/20 gives about **45 complete audits per quarter**. As an explicitly conditional illustration, applying the synthetic rates would yield about `45 * 47/916 = 2.3` die-crack cases and `45 * 21/916 = 1.0` multi-fault case per quarter. Rates need not transfer to operational data. [Brief](../problem-statement/helion_semiconductor_client_brief.md#business-context), [synthetic counts](../README.md#rejected-stack-cohort-and-seven-targets).

Pool qualified audit history rather than promising quarterly rare-fault guarantees. Report per-fault, 8-high/12-high, lot/time, tool/material, missingness and multi-fault slices where support permits. Pooling introduces process-change risk, so preserve versions and examine support across old/new conditions. Additional labels accumulate through the existing audit programme; the plan does not change the fixed metrology sampling rate.

### Cost accounting

```text
Total operational cost for the same cohort and follow-up
  = priced engineer attendance
    + priced equipment occupancy
    + setup/cleanup costs not already included above
    + consumables and separately justified consequences
    + approved delay penalties, only if explicitly adopted

Net value = incumbent cost - revised cost
            - incremental model, scheduler, coordination and qualification overhead
```

Turnaround and maximum waits are constraints and separate reports by default. A monetary delay penalty is optional only after agreement, with no second charge for the same consequence. Elapsed duration is not automatically engineer labour, and the quoted two-day cross-section is not 48 engineer-hours. Destructiveness first controls eligibility and evidence preservation; additional sample-preparation costs require data. Required acceptance costs common to all conditions cancel. [Original economics boundary](../README.md#where-diagnostic-test-savings-could-come-from).

The rolling-horizon planner optimises a limited current-work surrogate because future findings and routes are unknown. It is evaluated against expected total completion cost; it does not solve or prove the globally optimal diagnostic sequence. [Scheduling design](#helion-scheduling).


<a id="helion-operations"></a>
## Appendix E. Operating design and staged introduction

**Proposed behaviour:** score only cases with validated initial data after nightly consolidation. At the workstation, show eligible diagnostic alternatives and the proposed feasible resource/time assignment, with unresolved mechanisms, initial-score and resource-state timestamps, dependencies, cost assumptions and fallback reasons. The engineer confirms the test; a coordinator or designated shift lead confirms dispatch. Record overrides without forcing a choice. Preserve diagnostic evidence and resource commitments across handovers. Scheduling requires an additional current local operational feed, which is not supplied by the brief. The brief establishes nightly consolidation, air-gapped operation, rotating shifts and qualification. [Fab constraints](../problem-statement/helion_semiconductor_client_brief.md#business-context).

| Signal or event | Proposed trigger and action | Owner |
|---|---|---|
| Nightly import/scoring failure | Failure status, or no validated batch at scheduled handover: stop new ML recommendations, show SOP and recover the job | Fab IT with data owner |
| Case-level validation failure | Duplicate/conflicting key, required observation absent or invalid schema: hold that case from ML, retain reason and use SOP | Data owner; engineer continues SOP |
| Resource feed stale, scheduler failure or plan conflict | No new automated reservations; retain commitments/findings and use qualified manual dispatch. Reconcile before recovery | Coordinator/shift lead with fab IT |
| New result, arrival, outage, overrun or urgency change | Replan uncommitted future work with fresh state; preserve in-progress work and frozen reservations unless authorised exception | Coordinator/shift lead |
| Deadline, maximum-wait or due-work infeasible | Mark blocked reason and escalate; never silently postpone beyond the horizon or omit required expensive work | Coordinator and quality owner |
| Excess schedule changes or inaccurate durations | Review planned versus actual slots, queue age and changeover assumptions; adjust only through approved configuration | Scheduler owner and lab coordinator |
| New tool or supplier | Unqualified context appears: affected cases use SOP and enter qualification review | Process/quality owner with fab IT |
| Missingness or input shift | Compare coverage/tool/product distributions with qualified reference and pre-agreed control limits; investigate before calling it model decay | Yield Engineering |
| Inconclusive/repeated tests, overrides, open cases | Review weekly by supported slice and catalogue version; these are proxy signals, not ground truth | Quality owner and shift representatives |
| Audit completion and missed mechanisms | Track selected, pending, completed and lost cases; any audit-confirmed missed co-fault plausibly related to the recommendation suspends the affected policy for review | Quality owner |
| Candidate retraining | Confirmed model-related drift or qualified process change, AND owner-approved audited data/split/evaluation protocol | Model/data owner opens reviewed training request |
| Promotion | New candidate satisfies approved evidence gates and software qualification; preserve previous release for rollback | Quality owner and fab IT |

The handover clock time and statistical control limits must come from the actual MES schedule and a qualified reference period. Event-based failure triggers above are explicit defaults; we do not fabricate measured drift thresholds. Alerts identify an owner and a usable SOP path even when nobody is available to troubleshoot immediately at 3am.

**Release design:** package preprocessing, model, procedure catalogue, diagnostic-policy settings, scheduler configuration, cost definitions and resource-data contracts and a pinned offline CPU environment in a versioned manifest with hashes. Qualify a container runtime image if fab IT supports it. Keep signed-off installation, recovery and rollback instructions inside the fab. Save training and monitoring artifacts locally. No runtime calls to external services are needed.

**Introduction gates:** (1) retain existing SOP and manual dispatch while obtaining the closure standard, qualified catalogue, diagnostic logs and live scheduling contracts; (2) accumulate/version independent audit findings and qualify the data/software design; (3) conduct approved shadow evaluation; (4) run a controlled pilot only after the protocol and qualification gates are satisfied; (5) promote only on adequate completeness evidence, feasible turnaround/service obligations and net value. Qualify diagnostic and scheduling fallback combinations separately; a working scheduler may use SOP-derived candidates only if that mode is explicitly qualified. The present package completes the design step only. [Brief](../problem-statement/helion_semiconductor_client_brief.md#constraints--scope).


<a id="helion-worked-cases"></a>
## Appendix F. Worked cases

**Example versions:** the abstract cost units and capacity assumptions in this appendix are earlier, separate teaching illustrations. They are not SYN-OPS-001 dollar prices. Use [Appendix I](#helion-workflow-review) for the current USD costs, mock SOP, outcome assumptions and staff/equipment calendars used in the presentation.

**All cases and probabilities below are invented illustrations of the proposed policy.** They are not rows sampled from the CSVs, model predictions, validated diagnostic sensitivities or evidence of savings. Procedure duration/inconclusive inputs alone come from the [brief](../problem-statement/helion_semiconductor_client_brief.md#business-context).

### 1. Straightforward package investigation

Assume initial probabilities: warpage 0.70, underfill void 0.10, delamination 0.05, DRAM/base electrical 0.10, TSV 0.10, microbump 0.05, crack 0.02. All mechanisms start untested. No mandatory prerequisite blocks the candidate procedures.

Assume hypothetical comparable total costs of 100 units for X-ray, 200 for acoustic microscopy and 500 for electrical isolation at the candidate slots, with `w_m=1`. These invented costs stand in for a fully agreed non-overlapping cost calculation; they are not Helion rates. Using the brief's inconclusive fractions: X-ray covering warpage and underfill gives `0.90 * (0.70 + 0.10) / 100 = 0.0072`; acoustic gives `0.85 * (0.10 + 0.05) / 200 = 0.0006375`; electrical isolation gives `0.80 * (0.10 + 0.10) / 500 = 0.00032`. Scores are weighted priority per cost unit, not probabilities or measured savings. General delamination receives no X-ray credit without qualified gross-delamination evidence.

The policy passes eligible alternatives, including the higher-priority X-ray, to scheduling. The scheduler may propose another justified alternative if capacity, deadlines or setup-dependent costs change the feasible choice. The engineer checks diagnostic suitability and the coordinator confirms the slot. No observed queue or capacity has been invented as a dataset fact.

After a qualified positive warpage finding, only warpage becomes confirmed present. Other mechanisms remain unresolved until qualified evidence addresses them. Even if the actual hidden case has only warpage, the engineer does not know that merely from the first positive. Existing rules may choose the same X-ray, so this example demonstrates policy mechanics, not incremental ML value.

### 2. Coexisting TSV and microbump faults

Electrical isolation establishes TSV open. Microbump remains untested; its probability does not become zero. The SOP may require destructive cross-section after preservation of evidence and remaining prerequisite tests. If it confirms microbump, record both mechanisms. The engineer still checks the other outstanding mechanisms against the closure standard. This example directly guards against the brief's first-hit stopping failure.

### 3. Inconclusive acoustic examination

The test leaves delamination unresolved. Store the result, supported scope and reason. Do not label the mechanism absent or award the policy a completed investigation. Follow the approved alternative/repeat path and include all repeat effort. If that path is unavailable, abstain to the engineer's SOP escalation.

### 4. Missing measurements versus unavailable prerequisites

An unsampled detailed-metrology value is expected and can pass through the qualified imputation/coverage path. A missing acceptance record means the required prediction checkpoint cannot be established, so no ML recommendation is generated for that case. A true zero measurement remains zero. An untested fault label remains unknown. The same blank-looking cell can have different operational meanings.

### 5. Failed nightly import

At the agreed handover the batch has no validated completion. The interface shows the affected score availability, retains existing diagnostic findings and offers SOP. It never presents an old snapshot as a new score or delays mandatory testing. Fab IT and the data owner investigate, then validate recovery before restoring recommendations.

### 6. New bonder or underfill supplier

A new context falls outside qualification even if its measurements appear numerically familiar. Route affected cases to SOP, preserve supplier/tool provenance and inspect audit outcomes. A process change plus an approved new audited dataset/protocol opens candidate retraining. Qualification, rather than the drift alarm itself, authorises promotion. These are proposed responses to the brief's anticipated changes.


### 7. Two samples competing for equipment

**Entirely illustrative resource scenario:** one X-ray resource, one acoustic resource and sufficient qualified staff are available at time 0. Setup is zero in this simplified example, and resource occupancy equals the brief's stated 45/90-minute procedure durations. Stack A has an unresolved underfill question for which either test is assumed qualified. Stack B has an unresolved warpage question requiring X-ray. Assume evidence-resolution milestones of 120 minutes for A’s underfill question and 60 for B’s warpage question. Planned feasibility initially assumes the selected test gives conclusive qualified evidence for that question. These are invented planning limits, not client SLAs or complete-investigation guarantees.

If A takes X-ray first, B's planned completion is 90 minutes and misses its 60-minute limit. Schedule B on X-ray at 0–45, then A on X-ray at 45–90: both milestones can be met if results are conclusive. Another feasible plan uses B’s X-ray at 0–45 and A’s acoustic test at 0–90, but consumes 135 nominal procedure minutes instead of 90. Under these simplified constraints the acoustic alternative is not necessary. Prefer the cheaper qualified plan if the agreed total-cost comparison supports it; idle acoustic capacity alone does not justify extra work. The following case uses the acoustic branch only to demonstrate replanning. Any other unresolved mechanisms remain investigation obligations.

### 8. An inconclusive result changes the future schedule

Suppose A's acoustic result at minute 90 is inconclusive. Underfill remains unresolved; it is not a negative label. Recompute qualified alternatives and available slots. If no earlier qualified alternative remains available, a 45-minute X-ray starting at 90 finishes at 135, outside A’s illustrative 120-minute evidence-resolution milestone. Completing the inconclusive acoustic procedure did not satisfy that milestone. Flag the infeasibility and request coordinator/quality escalation rather than certify a deadline met, drop the case or change absence status. Other started work remains protected. Future unused conditional reservations can be released under the approved freeze policy.

### 9. A necessary expensive test reaches its maximum wait

Cross-section is required to resolve an outstanding mechanism after evidence-preservation prerequisites. New cheap cases repeatedly arrive. The mandatory-work and maximum-wait controls override the coverage/cost preference: allocate a feasible qualified slot or visibly escalate blocked capacity. An absent cost estimate disables numeric ranking, not the diagnostic obligation. Do not count moving the test beyond the planning horizon as savings.

### 10. Resource outage during committed work

An instrument goes unavailable after a plan is confirmed. Stop promising new slots on it, preserve findings and the reservation history, and let authorised staff handle safe interruption or specimen recovery. Replan unstarted work using qualified alternatives. Started non-preemptive procedures on unaffected equipment remain fixed. If live state cannot be trusted, use manual dispatch with the existing SOP rather than run an apparently feasible stale plan.


<a id="helion-traceability"></a>
## Appendix G. Coverage and source map

The [assessment rubric](../problem-statement/PRESENTATION_RUBRIC_APPRENTICE.md#2-dimensions) grades the design and individual live defence. The [official pre-read](../../../all-assignments/assignment8-ml_systems/PRE_READ_TEMPLATE.md) requires the completed notebook alongside its condensed answers.

| Rubric dimension | Notebook evidence | Presentation slide | Viva guide focus |
|---|---|---:|---|
| 1. Framing and fit | §§3, 10.2; appendices A/B | 1, 2, 4 | Two coordinated decisions, scope expansion, ML versus rules |
| 2. Cost of error and metric | §§3/6; appendix D | 5, 6 | Missing second fault, cost/turnaround/completeness trade-off |
| 3. Data and pipeline | §§4/5/6; appendices A/C | 3 | Provenance, partial labels, leakage, splits |
| 4. Serving and deployment | §7; appendices B/E | 7 | Nightly scores, live resources, rolling horizon and recovery |
| 5. Monitoring and loop | §8; appendix E | 7 | Audits, queues, replanning versus retraining |
| 6. Trade-offs | §§1/9/10.2; appendix D | 8 | Simplicity, limited evidence, economics |
| 7. Live defence | Worked cases and decision explanations throughout | All | Individual explanation, alternatives, what changes our mind |

**Answer-cell coverage:** §1 `cell008` and optional personal `cell010`; §2 `cell014`, `cell016`; §3 `cell021`, `cell023`, optional rules `cell025`; §4 `cell029`, `cell031`; §5 `cell036`, `cell038`, `cell040`; §6 `cell045`, `cell047`, `cell049`; §7 `cell053`, `cell055`, `cell057`; §8 `cell061`, `cell063`, `cell065`; §9 `cell069`, `cell071`; §10.2 `7a451b4c`; design exercises `130bb712`, `audit_write_test_a`. Original IDs are preserved in the notebook.

**Source precedence:** brief for scenario constraints; actual CSVs/dictionary for the delivered synthetic schema; README for archived/proposed task separation; course materials for submission requirements. A conflict is recorded rather than resolved by silently changing a meaning. Technical facts added to answers cite official Docker and scikit-learn documentation, accessed 18 September 2026. No external provider claim is treated as evidence that this Helion design works.

**Team completion:** supply member names/emails and any optional personal reflection, rehearse and challenge the authored reasoning, and obtain the course/client decisions explicitly marked unknown before claiming operational approval. No stakeholder agreement, mentor sign-off or production readiness is asserted.


**Broader-scope coverage:** [Appendix H](#helion-scheduling) defines alternatives/slots, rolling-horizon commitments, starvation prevention, operational data and scheduling failure handling. The presentation integrates scheduling throughout and the viva guide explains the original-scope trade-off. The 21 September revision changes authored answers and design appendices, while preserving original teaching questions and the recorded synthetic-data audit. The old source README/brief still describe the original proposal and are not silently rewritten.

**Workflow exercise, 21 September 2026:** [Appendix I](#helion-workflow-review) records a paper review of four scenarios, a prioritised evidence register, a bounded experiment specification and pipeline readiness decisions. Its outcomes are design findings, not engineer validation, model results or measured operational benefit. The [reading copy](workflow_walkthrough.md) is derived from the appendix.

**Costed revision:** [Appendix I](#helion-workflow-review) and its [reading copy](workflow_walkthrough.md) now include four costed scenarios, scoped report probabilities, exact resource phases, all-seven mock closure, exception paths and separate prediction/decision experiment specifications. The [presentation](presentation_outline.md) uses the same source versions and figures.


<a id="helion-scheduling"></a>
## Appendix H. Coupled diagnostic selection and rolling-horizon scheduling

This is an integrated proposed layer added at the user's request on 21 September 2026. It is not an implemented service or a claim that the original brief supplied scheduling data. The scope now includes two coordinated decisions, with one predictive model and a deterministic planner.

### Selection and scheduling exchange alternatives

1. Validate current case findings and resource state. Generate all technically justified next-test alternatives with coverage, prerequisites and mandatory obligations.
2. Evaluate feasible resource/time slots across the open cohort. Model equipment occupancy, staff skill/attendance, maintenance, specimen conflicts, setup/cleanup, batch compatibility and confirmed work separately. Precedence and non-overlap are standard resource-scheduling constraints. [OR-Tools job-shop formulation](https://developers.google.com/optimization/scheduling/job_shop).
3. Return slot-dependent delay, setup and incremental cost estimates to the diagnostic policy. Compare feasible alternatives in a bounded run; the coverage/cost ratio is advisory and cannot override completeness, deadlines or resource constraints.
4. Present one proposed procedure/resource/time choice for each dispatched case, plus alternatives and reasons. Engineer/coordinator confirmation commits only the chosen alternative, avoiding duplicate reservations for substitute tests.
5. Record actual start/completion, resources consumed, qualified findings and overrides. New information updates diagnosis and replans uncommitted future work.

### Rolling horizon and obligations

Use a configurable bounded planning horizon and qualified near-term freeze window. New arrivals, test results, equipment/staff changes, duration overruns and urgency changes trigger replanning. Protect started non-preemptive work and accepted frozen reservations; exceptions need authorised recovery/override records. The precise horizon, freeze window, duration buffers and service limits require operational qualification, not an invented calendar assumption.

Future diagnostic routes depend on outcomes, so initially commit the next justified procedures and reserve future contingent work only under explicit release rules. Do not treat every possible branch as certain work. Estimated task durations make planned deadline feasibility conditional; monitor actual overruns and escalate revised infeasibility instead of guaranteeing unknown completion times.

Minimising immediate cost must not produce an empty schedule or push expensive obligations beyond the horizon. Every due case receives a feasible action or an explicit blocked/escalated status. Carry terminal backlog and remaining closure obligations into the next plan and all evaluation reports. Apply approved maximum waits and ageing priorities so lower-probability or expensive necessary work cannot starve. Batching and fewer changeovers are allowed only within eligibility, audit, deadline and maximum-wait constraints.

Protect independently selected full-battery work and its completion monitoring even when it worsens apparent throughput. The scheduler cannot choose which cases supply ground truth. All pending and overdue audit cases remain in the denominator.

### Objective and practical approximation

The business objective concerns expected total investigation-completion cost. A deterministic planner of currently eligible actions cannot optimise every unknown future route. Use a transparent constrained rolling-horizon approximation, document its assumptions and evaluate actual completion outcomes. Do not claim the heuristic or planner is globally optimal.

Costs, queues and priorities are coupled: a preceding job changes setup, other reservations change waiting, and a different test choice changes both. Avoid endlessly reranking until an assumed convergence. Use a bounded planning budget, return the best verified feasible proposal available under the approved policy, and expose infeasibility or manual fallback when none is found. Any future solver must report feasible/optimal/timeout/infeasible distinctly; this document specifies behaviour without implementing a solver.

### Additional controls and verification scenarios

Check double-booked equipment, unavailable qualified staff, incompatible simultaneous work on one specimen, missing dependencies, stale calendars, duplicate substitute reservations, invalid setup transitions and batches that exceed a deadline. Also test outcome-driven replanning, preservation of commitments, necessary expensive work, overdue audit protection and explicit infeasibility. [Worked examples](#helion-worked-cases) illustrate these requirements without claiming runtime tests were executed.

Keep scheduling failures separate from ML degradation. A calendar outage needs operational recovery, not classifier retraining. A poor fault estimate needs model/evidence review, not an unexplained priority change. Releases and rollback include both policy and scheduler configurations, with humans retaining procedure, dispatch and closure authority.

**Current paper calendar:** [Appendix I](#helion-workflow-review) uses the one-day SYN-OPS-001 snapshot: staff phases are checked as well as machine occupancy. B-before-A CT appointments meet illustrative question milestones at the same $240 execution cost as A-before-B. Future SEM calendars remain unknown; a nominal 48-hour duration is not a promised completion.


<a id="helion-workflow-review"></a>
## Appendix I. A costed workflow walkthrough and next-step decisions

**21 September 2026 · Group 8 · Paper exercise using SYN-OPS-001**

This is the detailed record; the [standalone reading copy](workflow_walkthrough.md) is derived from this appendix. It completes steps 1–4 at design level: exercise decisions, prioritise evidence, specify experiments and choose what to build next. The scenarios are separate paper examples, not simultaneous bookings or observed investigations. No model, simulator, scheduler or operational pipeline has been implemented. Arithmetic and resource feasibility checks are not a performance benchmark.

### 0. What the new assumptions let us do

| Source | What it supplies | What it does not establish |
|---|---|---|
| [Client brief](../problem-statement/helion_semiconductor_client_brief.md) | Objective, procedure catalogue, nominal times, inconclusive rates, four Yield Engineering owners and independent audits | Actual full SOP, dollar rates or diagnostic staff calendars |
| [MOCK-ENG-001](mock_engineering_inspection_rules.md) | Invented triggers, fixed order, exceptions and all-seven-mechanism closure rule | Helion's actual approved procedure or closure standard |
| [SYN-OPS-001 assumptions](../synthetic-data-assumptions/operation-assumptions/synthetic_operating_assumptions.md) and [JSON](../synthetic-data-assumptions/operation-assumptions/synthetic_operating_assumptions.json) | Invented staff, equipment calendars, resource costs and outcome distributions | Observed costs, validated test performance or measured savings |
| Supplied CSV audit | 916 synthetic rejects, seven labels, existing lot/time splits | Procedure histories, diagnostic effort, audit membership or resource events |

We can now calculate concrete hypothetical decisions. Missing operational evidence remains missing; replacing a blank with an assumption does not verify it.

#### Use the same versioned operating inputs in every comparison

| Procedure | Resource cost per attempt, USD | Nominal duration from brief | Inconclusive probability |
|---|---:|---|---:|
| X-ray / CT | $120 | 45 minutes | 10% |
| Acoustic microscopy | $200 | 1.5 hours | 15% |
| Electrical isolation | $600 | 4 hours | 20% |
| Thermal / IR | $150 | 1 hour | 30% |
| Cross-section / SEM | $1,800 | 2 days, destructive | 5% |

Costs value technician labour at $70/h and diagnostic engineer labour at $110/h, plus equipment occupancy and consumables. Acoustic costs $35 technician + $55 engineer + $90 equipment + $20 supplies = $200. These are standard resource costs, not fully avoidable cash. Queue delay is not monetised, so waiting changes turnaround without automatically changing the dollar figure. SEM consumes six technician-hours and three engineer-hours; two days is not 48 attended labour-hours.

The invented shared laboratory has eight technicians and eight diagnostic engineers, separate from the four Yield Engineering owners in the brief. Headcount is not concurrent availability. At the 08:00 planning origin there are two day-shift technicians and one engineer; TECH-01 and QE-01 are occupied until 09:00, TECH-02 until 10:00. CT is reserved until 09:00 and electrical isolation until 12:00. The acoustic machine is free at 08:00, but its qualified preparation staff are not. Availability beyond the first 24 hours is unknown.

#### Execute a decision event in this order

1. Check case identity, initial observations, current evidence, audit membership and source versions. Missing evidence stays unknown.
2. Apply mock triggers and remaining completeness obligations. Determine eligible procedures and preserve intact-sample evidence before destructive work.
3. Use the seven initial probabilities as optional priority inputs. They do not sum to one and are not updated posteriors after findings.
4. Check qualified staff phases, equipment, specimen conflicts and existing reservations. Return a feasible proposed slot or a blocked status.
5. Engineer approves the procedure; the designated dispatcher confirms the slot. Record resource use even if the result is inconclusive.
6. Update only questions supported by the report's scope. Reconsider future work; close only after the mock evidence standard is met and the engineer reviews it.

Track W = warpage, V = underfill void, D = delamination, E = DRAM/base electrical, T = TSV, M = microbump and C = die crack. Each is untested, confirmed present, confirmed absent or inconclusive. Untested/inconclusive remain unresolved. Here “confirmed” describes the mock evidence record: reports are fallible, and independent evaluation must still assess diagnostic errors against hidden truth. Process engineers use findings to investigate manufacturing causes; a delamination finding alone does not identify a particular faulty machine or recipe.

### 1. Exercise the decision workflow

#### Case A: a strong rule, a priced test and a feasible slot

Assume an 8-high rejected stack has warpage 17 µm, so mock R01 fires at its invented 15 µm threshold. No competing initial trigger fires. Initial W/V/D/E/T/M/C probabilities are illustratively 0.70/0.10/0.05/0.10/0.10/0.05/0.02. The mock baseline and proposed policy both select CT.

| Checkpoint | Decision / evidence | Cost and pending work |
|---|---|---|
| 08:00 request | CT cannot start: machine and preparation staff occupied | No new attempt cost; one hour planned waiting |
| 09:00–09:45 | CT reserved; TECH-01 attends 09:00–09:15; QE-01 interprets 09:30–09:45 | $120 attempt; no resource overlap |
| Branch A1: qualified W positive, V negative, gross-D negative | Record W present and V absent; gross-D negative cannot exclude all D | D/E/T/M/C unresolved; closure not allowed |
| Branch A2: qualified W/V negative, gross-D negative | Record W/V absent | D/E/T/M/C unresolved; continue completeness review |
| Branch A3: procedure inconclusive | W/V/D remain unresolved | $120 still spent; review scoped alternative/repeat eligibility |

Branches are alternative reports, not three attempts. Under SYN-OPS-001, an inconclusive event is shared across questions examined by that procedure. After A1/A2, acoustic is the next mock completeness procedure for D; required electrical and microbump/crack work remains visible. A3 does not automatically rerun CT: acoustic can address V/D, while unresolved W needs a reviewed plan.

**Conclusion:** the rule already makes this choice. Pricing and scheduling make its consequences explicit; agreement does not show incremental ML value.

#### Case B: acoustic-first can be correct and already match the baseline

Assume a prior qualified CT has settled W and V as absent and cost $120; its gross-D negative leaves D unresolved. Mock R03 fires from `delamination_area_pct = 0.40`, while R04 fires from `detected_interconnect_failures = 1`. The current illustrative D/E/T probabilities are 0.70/0.10/0.10. Required nondestructive work remains before SEM.

Both acoustic and electrical isolation are justified. **MOCK-ENG-001 chooses acoustic first**, because its fixed order puts acoustic before electrical among eligible procedures. The brief's electrical example does not establish a universal electrical-first rule. CT has no unresolved W/V question here and is not an automatic repeat candidate.

For comparison only, the earlier positive-coverage heuristic with equal weights gives:

| Eligible procedure | Calculation using current unresolved questions | Priority per resource dollar |
|---|---|---:|
| Acoustic | 0.85 × 0.70 / 200 | 0.002975 |
| Electrical isolation | 0.80 × (0.10 + 0.10) / 600 | 0.000267 |

This heuristic agrees with the mock baseline. It is not a probability of completing the investigation, does not fully value negative evidence and does not model the listed sensitivity/specificity. It is a candidate to evaluate, not an established cost-minimising policy.

| Action / illustrative report | Feasible slot in this separate scenario | Cumulative resource cost | Questions still unresolved |
|---|---|---:|---|
| Prior CT: W/V absent | Already completed before this snapshot | $120 | D/E/T/M/C |
| Acoustic: D present; no contradictory V finding | 09:00–10:30; TECH-01 09:00–09:30, QE-01 10:00–10:30 | $320 | E/T/M/C |
| Electrical: E absent, T present | 12:00–16:00; TECH-01 12:00–12:30, QE-01 13:30–16:00 | $920 | M/C |
| SEM requested after evidence preservation | Future qualified staff/calendar needed; no completion slot promised | $920 spent; $1,800 planned | M/C remain pending |

The real D finding is useful. Comparing the $800 acoustic-plus-electrical pair against $600 electrical-only would compare different evidence: electrical does not settle D. If both procedures are needed, either order costs $800. The $120 CT already performed is sunk for the next-action choice, but stays in the full-investigation total. Pending SEM is not counted as spent or omitted from planned obligations.

**Outcome probability is a separate calculation.** Using the invented acoustic sensitivity 0.97 and specificity 0.98 conditional on a conclusive examination, q = 0.15 and illustrative prior p(D) = 0.70:

- Positive report: `(1 − 0.15) × [0.70 × 0.97 + 0.30 × 0.02] = 58.225%`.
- Negative report: `(1 − 0.15) × [0.70 × 0.03 + 0.30 × 0.98] = 26.775%`.
- Inconclusive report: `15%`.

These sum to one. The 70% fault prior, 85% conclusiveness and 58.225% positive-report probability answer different questions. Neither positive nor negative guarantees truth. If acoustic is inconclusive, $320 has still been spent, D stays unresolved and a reviewed continuation is needed. Independent electrical work can proceed at its feasible slot; it cannot close D. These report probabilities alone do not give the cost of that continuation.

**Conclusion:** this case supports a useful acoustic examination, but no change in choice or saving attributable to ML.

#### Case C: complete the evidence record, including the second fault

Continue Case B's conclusive branch after the lab obtains future calendars, preserves all required nondestructive evidence and authorises SEM. Its low initial microbump probability, say 0.05, cannot remove the unresolved M question. Assume a scoped conclusive SEM report confirms M and T and excludes C, without contradictions.

| Mechanism | Final mock evidence state | Supporting report |
|---|---|---|
| W | Confirmed absent | CT |
| V | Confirmed absent | CT; acoustic consistent |
| D | Confirmed present | Acoustic |
| E | Confirmed absent | Electrical isolation |
| T | Confirmed present | Electrical isolation and SEM |
| M | Confirmed present | SEM |
| C | Confirmed absent | SEM |

The engineer can now review closure against the mock all-seven standard. Total resource cost for this particular path is **$120 + $200 + $600 + $1,800 = $2,720**. SEM's invented package-wide exclusion scope is a strong assumption; a limited real section cannot automatically support C absence. Hidden truth, kept from the policy, is needed to assess whether the recorded findings are actually correct. This branch is a scripted illustration, not a simulated success rate.

An inconclusive SEM instead leaves M/C unresolved: the same $2,720 has been consumed without complete closure. Escalate the evidence gap; no automatic repeated destructive sampling or first-positive stopping. IR would add $150 when required by R07 or the independent audit battery; it does not directly establish a mechanism's absence. Running all five once costs $2,870 and still does not guarantee complete or correct findings.

Not every case requires all four core procedures: a qualified gross-D positive on CT may already settle D, and a procedure with no remaining required question may be skipped under the mock rules. CT + electrical + SEM would then cost $2,520 if no acoustic or IR obligation remains. Audits still require their battery. Any such omission must be available to the baseline too; $2,720 is a costed branch, not a universal minimum or fixed cost for every investigation.

#### Case D: staffing and deadlines change scheduling, not fault probability

At the same 08:00 origin, two separate rejected stacks need CT: A has an underfill question with a milestone of 11:00; B has a warpage question due at 10:00. These are invented deadlines for scoped evidence, not whole-investigation closure. CT and TECH-01 first become available at 09:00. The described examinations are eligible; other obligations remain pending.

| Plan | First CT | Second CT | Milestone assessment if conclusive | Total resource cost |
|---|---|---|---|---:|
| A first | A 09:00–09:45 | B 09:45–10:30 | B misses 10:00 | $240 |
| B first | B 09:00–09:45 | A 09:45–10:30 | Both met | $240 |

For B-first, TECH-01 is needed 09:00–09:15 and 09:45–10:00; QE-01 is needed 09:30–09:45 and 10:15–10:30. These intervals fit the supplied staff commitments and do not overlap for either person. This checks more than machine availability. The model's initial fault probabilities need not change.

If B is inconclusive at 09:45, B remains unresolved and the $120 attempt is retained. An automatic repeat is not authorised. Even a hypothetically approved immediate 45-minute repeat cannot finish by 10:00 and would conflict with A's accepted slot. Preserve A's commitment and escalate B's milestone/continuation. Stale calendars or outages likewise require manual reconciliation, not invented availability.

Choosing acoustic for A solely because the machine is free would cost $200 rather than $120 and requires qualified staff; the B-first CT plan already meets both scoped milestones. No additional procedure is justified by machine idleness alone. Actual lab dispatch might already use B-first, so this is a feasibility demonstration, not a measured improvement.

#### Exception walkthroughs

| Situation | Immediate action | Cost, evidence and continuation |
|---|---|---|
| Required measurement missing | Mark the affected trigger not evaluable; preserve missingness reason; continue other supported branches | Never fill with zero or infer absence; unresolved questions remain |
| Nightly import fails | Use existing rules and last valid case evidence; do not create fresh model scores | Manual dispatch if resource state is also unqualified; keep audits and obligations |
| Resource calendar stale or beyond 24-hour horizon | Stop new automatic slot promises; request current local availability | Keep consumed cost, findings and existing commitments; SEM's nominal 48 hours is not a guaranteed finish |
| New bonder or supplier | Flag the case outside qualified conditions; use reviewed SOP/manual handling and qualification review | Neither old thresholds nor invented sensitivities establish performance on the new process |

### 2. Prioritise the evidence still needed

The assumptions resolve a teaching exercise's inputs; they do not close the real evidence gaps. Requests below are specifications, not messages sent or client answers obtained.

| Topic | Available for the exercise | Evidence needed before operational claims; owner |
|---|---|---|
| Baseline / closure | MOCK-ENG-001 with seven separate states | Actual SOP, exceptions, closure and repeat standard; quality owner |
| Costs | SYN-OPS-001 staff/equipment rates and phases | Actual labour, occupancy, consumables and cost boundaries; finance/lab operations |
| Availability | One-day hypothetical roster and reservations | Qualified skills, live calendars, deadlines, setup/batching and current dispatch; shift lead |
| Test outcomes | Invented scoped sensitivity/specificity and brief q | Validated coverage, correlated errors, result histories and destructive limits; lab/quality |
| Fault prediction | 916 synthetic labels and existing splits | Decision-time availability, representative complete audits and fresh process evidence; data/quality |

Capture event time and information-availability time, initial inputs, evidence states, pending obligations, eligible alternatives, baseline and proposed choices, reasons/overrides, staff/equipment assignments, scoped reports, actual resource use and closure. Keep independent audit selection, pending audits and unresolved backlog visible. Do not request more metrology sampling or convert selectively unobserved labels into negatives. [Notebook evidence](P1_ml_systems_helion.ipynb#helion-evidence)

### 3. Specify two different experiments

**Prediction experiment, optional and not run:** ask whether allowed manufacturing context improves fault probabilities beyond inspection/acceptance inputs. Retain the 653/126/137 rejected lot/time partitions, training-only preprocessing, per-fault support/calibration and lot-aware uncertainty. Compare training prevalence, inspection-only logistic and the same regularised model family with allowed context. Exclude later annotations, simulator internals and `timing_margin_ps`. Previously examined synthetic holdout data supplies retrospective evidence only. The two logistic configurations are alternatives, not two deployed models.

**Decision experiment, proposed next:** exercise the mock rules, the earlier coverage heuristic and feasible schedules against identical operating assumptions. Use scripted branches before stochastic simulation. Fault priors are not procedure outcome labels; synthetic sensitivities are not a trained outcome predictor. Store hidden truth separately from policy inputs, preserve coexisting faults, and do not assume independent repeat-test successes. Unspecified continuation branches end in explicit unresolved/escalated states, not invented cost-free completion.

| Arm | Diagnostic selection | Dispatch |
|---|---|---|
| A | Mock engineering rules | Explicit synthetic reference dispatch |
| B | Qualified candidate policy | Same reference dispatch |
| C | Mock engineering rules | Candidate constrained dispatch |
| D | Candidate policy | Candidate constrained dispatch |

Actual Helion dispatch remains unknown. For Case D alone, A-first versus B-first is an explicit teaching contrast, not the real incumbent. A later broader reference dispatch must be specified and frozen before comparison. Separate selection, scheduling and their interaction. Include a rules-plus-evidence-checklist alternative so workflow discipline is not credited to ML.

A better decision objective than a greedy ratio is **immediate resource cost plus expected remaining cost to adequate completion**, subject to resource and evidence constraints. The cost table supplies the first term; scoped report probabilities supply some branches. The complete continuation model is not yet specified for every inconclusive result, so we cannot calculate a globally optimal policy or total expected savings from these inputs alone. The initial model remains the seven-output logistic candidate; no automatic Bayesian update or additional learned outcome model is claimed.

Report per-investigation resource cost, labour/equipment use, elapsed mean/p95 turnaround, missed milestones, overrides, pending cost/obligations, unresolved cases and missed mechanisms/co-faults. Compare cost at the same adequacy and diagnostic-error requirements; do not drop unfinished cases. Sensitivity-check rates ×0.75/1.25, equipment outages of 2/4 hours, q ±0.05 and worse sensitivity/specificity as defined in SYN-OPS-001. These are stress tests, not confidence intervals. A fixed same-test path has the same execution cost regardless of order; no current batch/setup or queue-dollar saving is assumed.

### 4. What to build next

| Stage | Concrete output | Gate / limitation |
|---|---|---|
| Completed here | Versioned, costed paper cases and presentation | Internally checked assumptions; no operational validation |
| Next educational step | Small deterministic replay of these cases, then a bounded stochastic simulator if useful | Freeze scope, continuation rules, comparison dispatch and evaluation metrics first; preserve unresolved endings |
| Optional prediction study | Inspection-versus-context experiment above | Demonstrate added predictive information without claiming operational savings |
| Operational discovery | Replace assumptions with reviewed SOP, rates, event history and resource feeds | Independent full-battery evidence and representative process coverage |
| Qualified shadow study | Record recommendations while engineers follow current practice | Qualified interfaces, fallbacks and monitoring; no realised-benefit claim from shadow recommendations alone |
| Controlled pilot / release | Evaluate justified changes with common follow-up and backlog accounting | Evidence of benefit at unchanged adequacy; reviewed release, never automatic deployment |

**Current decision:** the assumptions are sufficient for a concrete learning demonstration. Build that bounded replay before a production model pipeline if implementation is requested. Retain the option to use rules with better scheduling, or retain existing practice if additional complexity earns no benefit. Neither the new prices nor accurate fault predictions alone prove that a cheaper complete diagnostic path exists.
